In [ ]:
print("Quantum Disaster Response System")
print("Starting project setup...")


In [ ]:
!pip install -q fastapi uvicorn requests pandas numpy scikit-learn joblib
!pip install -q qiskit==2.5.2 qiskit-aer==0.17.2 qiskit-optimization==0.7.0
!pip install -q global-land-mask

In [ ]:
import qiskit
import qiskit_aer
import qiskit_optimization
import sklearn

print("Qiskit:", qiskit.__version__)
print("Qiskit Aer:", qiskit_aer.__version__)
print("Qiskit Optimization:", qiskit_optimization.__version__)
print("Scikit-learn:", sklearn.__version__)
print("All packages installed successfully.")

In [ ]:
from pathlib import Path

BASE = Path("/content/quantum-disaster-response")
BACKEND = BASE / "backend"
SERVICES = BACKEND / "services"

BACKEND.mkdir(parents=True, exist_ok=True)
SERVICES.mkdir(parents=True, exist_ok=True)

print("Project created:")
print(BASE)
print(BACKEND)
print(SERVICES)


In [ ]:
(SERVICES / "__init__.py").write_text("")
print("services/__init__.py created")

In [ ]:
usgs_code = r'''
import requests

USGS_URL = (
    "https://earthquake.usgs.gov/"
    "earthquakes/feed/v1.0/summary/all_day.geojson"
)


def get_earthquakes():

    response = requests.get(
        USGS_URL,
        timeout=15
    )

    response.raise_for_status()

    data = response.json()

    earthquakes = []

    for feature in data.get("features", []):

        properties = feature.get(
            "properties",
            {}
        )

        geometry = feature.get(
            "geometry",
            {}
        )

        coordinates = geometry.get(
            "coordinates",
            []
        )

        if len(coordinates) < 3:
            continue

        earthquake = {

            "id": feature.get("id"),

            "source": "USGS",

            "type": "Earthquake",

            "title": properties.get(
                "title"
            ),

            "magnitude": properties.get(
                "mag"
            ),

            "place": properties.get(
                "place"
            ),

            "time": properties.get(
                "time"
            ),

            "longitude": coordinates[0],

            "latitude": coordinates[1],

            "depth": coordinates[2],

            "url": properties.get(
                "url"
            )
        }

        earthquakes.append(
            earthquake
        )

    return earthquakes
'''

(SERVICES / "usgs_service.py").write_text(usgs_code)

print("usgs_service.py created")

In [ ]:
eonet_code = r'''
import requests

EONET_URL = (
    "https://eonet.gsfc.nasa.gov/api/v3/events"
)


def get_eonet_events():

    params = {
        "status": "open",
        "limit": 50
    }

    response = requests.get(
        EONET_URL,
        params=params,
        timeout=15
    )

    response.raise_for_status()

    data = response.json()

    events = []

    for event in data.get(
        "events",
        []
    ):

        categories = event.get(
            "categories",
            []
        )

        if categories:

            category = categories[0].get(
                "title",
                "Unknown"
            )

        else:

            category = "Unknown"

        geometry_list = event.get(
            "geometry",
            []
        )

        if not geometry_list:
            continue

        geometry = geometry_list[-1]

        coordinates = geometry.get(
            "coordinates"
        )

        if not coordinates:
            continue

        if geometry.get("type") != "Point":
            continue

        event_data = {

            "id": event.get("id"),

            "source": "NASA EONET",

            "type": category,

            "title": event.get(
                "title"
            ),

            "date": geometry.get(
                "date"
            ),

            "longitude": coordinates[0],

            "latitude": coordinates[1],

            "url": event.get(
                "link"
            )
        }

        events.append(
            event_data
        )

    return events
'''

(SERVICES / "eonet_service.py").write_text(eonet_code)

print("eonet_service.py created")

In [ ]:
feature_code = r'''
def normalize(value, minimum, maximum):

    if maximum == minimum:
        return 0

    value = (
        value - minimum
    ) / (
        maximum - minimum
    )

    return max(
        0,
        min(1, value)
    )


def calculate_earthquake_features(event):

    magnitude = event.get(
        "magnitude"
    )

    depth = event.get(
        "depth"
    )

    if magnitude is None:
        magnitude = 0

    if depth is None:
        depth = 100

    severity = (
        magnitude / 10
    ) * 100

    depth_factor = max(
        0,
        1 - depth / 200
    )

    severity = (
        severity * 0.8
        + depth_factor * 20
    )

    severity = max(
        0,
        min(100, severity)
    )

    people = min(
        100,
        severity * 0.9
    )

    medical = min(
        100,
        severity * 0.85
    )

    rescue = min(
        100,
        severity * 0.9
    )

    return {

        "severity": round(
            severity,
            2
        ),

        "estimated_people_affected":
            round(
                people,
                2
            ),

        "estimated_medical_need":
            round(
                medical,
                2
            ),

        "estimated_rescue_need":
            round(
                rescue,
                2
            )
    }


def calculate_generic_features(event):

    severity = 50

    event_type = event.get(
        "type",
        ""
    ).lower()

    title = event.get(
        "title",
        ""
    ).lower()

    if "wildfire" in event_type:

        severity = 70

    elif (
        "storm" in event_type
        or
        "cyclone" in event_type
    ):

        severity = 72

    elif "flood" in event_type:

        severity = 78

    elif "volcano" in event_type:

        severity = 85

    elif "landslide" in event_type:

        severity = 68

    if any(
        word in title
        for word in [
            "super typhoon",
            "major hurricane",
            "category 5",
            "cat 5"
        ]
    ):

        severity += 12

    elif any(
        word in title
        for word in [
            "typhoon",
            "hurricane",
            "cyclone"
        ]
    ):

        severity += 7

    elif any(
        word in title
        for word in [
            "severe",
            "extreme",
            "major"
        ]
    ):

        severity += 5

    elif any(
        word in title
        for word in [
            "tropical depression",
            "weak"
        ]
    ):

        severity -= 5

    severity = max(
        0,
        min(100, severity)
    )

    medical = min(
        100,
        severity * 0.80
    )

    rescue = min(
        100,
        severity * 0.85
    )

    people = min(
        100,
        severity * 0.75
    )

    return {

        "severity":
            round(
                severity,
                2
            ),

        "estimated_people_affected":
            round(
                people,
                2
            ),

        "estimated_medical_need":
            round(
                medical,
                2
            ),

        "estimated_rescue_need":
            round(
                rescue,
                2
            )
    }


def calculate_resource_requirements(event):

    severity = event.get(
        "severity",
        0
    )

    medical_need = event.get(
        "estimated_medical_need",
        0
    )

    rescue_need = event.get(
        "estimated_rescue_need",
        0
    )

    rescue_teams_needed = max(
        1,
        min(
            3,
            round(
                rescue_need / 35
            )
        )
    )

    medical_units_needed = max(
        1,
        min(
            4,
            round(
                medical_need / 25
            )
        )
    )

    relief_units_needed = max(
        1,
        min(
            4,
            round(
                severity / 25
            )
        )
    )

    response_time_minutes = round(
        120 - severity,
        2
    )

    travel_distance_km = round(
        500 - severity * 4,
        2
    )

    return {

        "estimated_rescue_teams_needed":
            rescue_teams_needed,

        "estimated_medical_units_needed":
            medical_units_needed,

        "estimated_relief_units_needed":
            relief_units_needed,

        "estimated_response_time_minutes":
            response_time_minutes,

        "estimated_travel_distance_km":
            travel_distance_km
    }


def add_features(event):

    if event.get("source") == "USGS":

        features = calculate_earthquake_features(
            event
        )

    else:

        features = calculate_generic_features(
            event
        )

    event.update(
        features
    )

    event.update(
        calculate_resource_requirements(
            event
        )
    )

    return event
'''

(SERVICES / "feature_service.py").write_text(feature_code)

print("feature_service.py created")

In [ ]:
ml_code = r'''
import numpy as np

from sklearn.ensemble import RandomForestRegressor


class DisasterMLModel:

    def __init__(self):

        self.model = RandomForestRegressor(
            n_estimators=150,
            random_state=42
        )

        self.train_model()


    def train_model(self):

        rng = np.random.default_rng(
            42
        )

        X = []

        y = []

        for _ in range(1000):

            severity = rng.uniform(
                0,
                100
            )

            people = rng.uniform(
                0,
                100
            )

            medical = rng.uniform(
                0,
                100
            )

            rescue = rng.uniform(
                0,
                100
            )

            priority = (

                0.35 * severity

                + 0.30 * people

                + 0.15 * medical

                + 0.15 * rescue

                + 0.05 *
                rng.uniform(
                    0,
                    100
                )
            )

            X.append([
                severity,
                people,
                medical,
                rescue
            ])

            y.append(
                priority
            )

        X = np.array(X)

        y = np.array(y)

        self.model.fit(
            X,
            y
        )


    def predict(self, event):

        X = np.array([

            [

                event.get(
                    "severity",
                    0
                ),

                event.get(
                    "estimated_people_affected",
                    0
                ),

                event.get(
                    "estimated_medical_need",
                    0
                ),

                event.get(
                    "estimated_rescue_need",
                    0
                )

            ]

        ])

        prediction = (
            self.model.predict(X)[0]
        )

        prediction = max(
            0,
            min(
                100,
                prediction
            )
        )

        return round(
            float(prediction),
            2
        )


ml_model = DisasterMLModel()


def predict_priority(event):

    score = ml_model.predict(
        event
    )

    if score >= 75:

        level = "CRITICAL"

    elif score >= 50:

        level = "HIGH"

    elif score >= 25:

        level = "MEDIUM"

    else:

        level = "LOW"

    event["priority_score"] = score

    event["priority_level"] = level

    return event
'''

(SERVICES / "ml_service.py").write_text(ml_code)

print("ml_service.py created")

In [ ]:
quantum_code = r'''
import time

from qiskit.primitives import StatevectorSampler

from qiskit_optimization import QuadraticProgram

from qiskit_optimization.algorithms import (
    MinimumEigenOptimizer
)

from qiskit_optimization.minimum_eigensolvers import (
    QAOA
)

from qiskit_optimization.optimizers import (
    COBYLA
)


def quantum_allocate(
    disasters,
    available_teams=3,
    available_medical_units=5,
    available_relief_units=5
):

    if not disasters:

        return {

            "selected": [],

            "objective": 0,

            "teams_used": 0,

            "medical_units_used": 0,

            "relief_units_used": 0,

            "available_teams":
                available_teams,

            "available_medical_units":
                available_medical_units,

            "available_relief_units":
                available_relief_units,

            "method": "QAOA",

            "status": "NO_DATA"
        }


    candidates = sorted(

        disasters,

        key=lambda x:
            x.get(
                "priority_score",
                0
            ),

        reverse=True

    )[:5]


    n = len(candidates)

    qp = QuadraticProgram()


    for i in range(n):

        qp.binary_var(
            name=f"x{i}"
        )


    values = {}


    for i, disaster in enumerate(
        candidates
    ):

        priority = disaster.get(
            "priority_score",
            0
        )

        medical_need = disaster.get(
            "estimated_medical_need",
            0
        )

        rescue_need = disaster.get(
            "estimated_rescue_need",
            0
        )

        response_time = disaster.get(
            "estimated_response_time_minutes",
            120
        )

        travel_distance = disaster.get(
            "estimated_travel_distance_km",
            500
        )

        value = (

            0.50 * priority

            + 0.20 * medical_need

            + 0.20 * rescue_need

            - 0.05 * response_time

            - 0.05 *
            travel_distance / 5
        )

        values[i] = value


    qp.maximize(

        linear={

            f"x{i}":
                values[i]

            for i in range(n)

        }

    )


    qp.linear_constraint(

        linear={

            f"x{i}":
                candidates[i].get(
                    "estimated_rescue_teams_needed",
                    1
                )

            for i in range(n)

        },

        sense="<=",

        rhs=available_teams,

        name="rescue_team_limit"
    )


    qp.linear_constraint(

        linear={

            f"x{i}":
                candidates[i].get(
                    "estimated_medical_units_needed",
                    1
                )

            for i in range(n)

        },

        sense="<=",

        rhs=available_medical_units,

        name="medical_unit_limit"
    )


    qp.linear_constraint(

        linear={

            f"x{i}":
                candidates[i].get(
                    "estimated_relief_units_needed",
                    1
                )

            for i in range(n)

        },

        sense="<=",

        rhs=available_relief_units,

        name="relief_unit_limit"
    )


    start_time = time.time()


    qaoa = QAOA(

        sampler=StatevectorSampler(
            seed=42
        ),

        optimizer=COBYLA(
            maxiter=100
        ),

        reps=1
    )


    optimizer = MinimumEigenOptimizer(
        qaoa
    )


    result = optimizer.solve(
        qp
    )


    runtime = (
        time.time()
        - start_time
    )


    selected = []


    for i, value in enumerate(
        result.x
    ):

        if round(value) == 1:

            selected.append(
                candidates[i]
            )


    teams_used = sum(

        d.get(
            "estimated_rescue_teams_needed",
            1
        )

        for d in selected

    )


    medical_units_used = sum(

        d.get(
            "estimated_medical_units_needed",
            1
        )

        for d in selected

    )


    relief_units_used = sum(

        d.get(
            "estimated_relief_units_needed",
            1
        )

        for d in selected

    )


    objective = sum(

        values[i]

        for i, value in enumerate(
            result.x
        )

        if round(value) == 1

    )


    return {

        "selected": selected,

        "objective":
            round(
                float(objective),
                2
            ),

        "teams_used":
            teams_used,

        "medical_units_used":
            medical_units_used,

        "relief_units_used":
            relief_units_used,

        "available_teams":
            available_teams,

        "available_medical_units":
            available_medical_units,

        "available_relief_units":
            available_relief_units,

        "candidate_count":
            n,

        "runtime_seconds":
            round(
                runtime,
                4
            ),

        "method":
            "QAOA",

        "status":
            "SUCCESS"
    }
'''

(SERVICES / "quantum_service.py").write_text(quantum_code)

print("quantum_service.py created")

In [ ]:
location_code = r'''
import requests

CACHE = {}


def get_location_name(
    latitude,
    longitude
):

    if latitude is None or longitude is None:

        return "Location unavailable"


    try:

        latitude = float(latitude)

        longitude = float(longitude)

    except (
        TypeError,
        ValueError
    ):

        return "Location unavailable"


    key = (
        round(latitude, 3),
        round(longitude, 3)
    )


    if key in CACHE:

        return CACHE[key]


    url = (
        "https://nominatim.openstreetmap.org/reverse"
    )


    params = {

        "lat": latitude,

        "lon": longitude,

        "format": "json",

        "zoom": 10,

        "addressdetails": 1
    }


    headers = {

        "User-Agent":
            "QuantumDisasterResponse/1.0"
    }


    try:

        response = requests.get(

            url,

            params=params,

            headers=headers,

            timeout=15
        )

        response.raise_for_status()

        data = response.json()

        address = data.get(
            "address",
            {}
        )


        city = (

            address.get("city")

            or address.get(
                "town"
            )

            or address.get(
                "municipality"
            )

            or address.get(
                "village"
            )

            or address.get(
                "county"
            )

            or "Unknown City"
        )


        state = (

            address.get(
                "state"
            )

            or "Unknown State"
        )


        country = (

            address.get(
                "country"
            )

            or "Unknown Country"
        )


        location = (
            f"{city}, "
            f"{state}, "
            f"{country}"
        )


    except Exception:

        location = (
            f"Coordinates: "
            f"{latitude:.3f}, "
            f"{longitude:.3f}"
        )


    CACHE[key] = location

    return location
'''

(SERVICES / "location_service.py").write_text(
    location_code
)

print("location_service.py created")

In [ ]:
routing_code = r'''

import requests

OVERPASS_URL = "https://overpass-api.de/api/interpreter"
OSRM_TABLE_URL = "https://router.project-osrm.org/table/v1/driving"

ROUTE_CACHE = {}
FACILITY_CACHE = {}


def _valid_coordinates(latitude, longitude):
    try:
        latitude = float(latitude)
        longitude = float(longitude)
    except (TypeError, ValueError):
        return False

    return (
        -90 <= latitude <= 90
        and -180 <= longitude <= 180
    )


def _get_emergency_facilities(latitude, longitude, radius_m=50000):
    """
    Find real mapped emergency facilities near the disaster.
    OpenStreetMap provides facility locations/types, not live team counts.
    """
    if not _valid_coordinates(latitude, longitude):
        return []

    latitude = float(latitude)
    longitude = float(longitude)

    key = (
        round(latitude, 2),
        round(longitude, 2),
        radius_m
    )

    if key in FACILITY_CACHE:
        return FACILITY_CACHE[key]

    query = f"""
    [out:json][timeout:25];
    (
      nwr(around:{radius_m},{latitude},{longitude})["amenity"="hospital"];
      nwr(around:{radius_m},{latitude},{longitude})["amenity"="fire_station"];
      nwr(around:{radius_m},{latitude},{longitude})["emergency"="disaster_response"];
    );
    out center tags;
    """

    try:
        response = requests.post(
            OVERPASS_URL,
            data=query,
            timeout=30,
            headers={
                "User-Agent":
                    "QuantumDisasterResponse/1.0"
            }
        )
        response.raise_for_status()
        elements = response.json().get("elements", [])
    except Exception:
        FACILITY_CACHE[key] = []
        return []

    facilities = []

    for element in elements:
        tags = element.get("tags", {})

        if "lat" in element and "lon" in element:
            lat = element["lat"]
            lon = element["lon"]
        else:
            center = element.get("center", {})
            lat = center.get("lat")
            lon = center.get("lon")

        if lat is None or lon is None:
            continue

        amenity = tags.get("amenity")
        emergency = tags.get("emergency")

        if amenity == "hospital":
            facility_type = "Hospital"
        elif amenity == "fire_station":
            facility_type = "Fire Station"
        elif emergency == "disaster_response":
            facility_type = "Disaster Response Center"
        else:
            facility_type = "Emergency Facility"

        facilities.append({
            "id": f"osm_{element.get('type')}_{element.get('id')}",
            "name": (
                tags.get("name")
                or tags.get("official_name")
                or facility_type
            ),
            "type": facility_type,
            "latitude": float(lat),
            "longitude": float(lon)
        })

    # Remove duplicates and keep a manageable set.
    unique = {}
    for facility in facilities:
        key = (
            round(facility["latitude"], 4),
            round(facility["longitude"], 4),
            facility["type"]
        )
        unique[key] = facility

    facilities = list(unique.values())[:12]
    FACILITY_CACHE[key] = facilities
    return facilities


def calculate_actual_route_to_response_center(event):
    """
    Calculate real road-network distance and route duration to the
    nearest mapped emergency facility using OSRM.

    The duration is a route-based travel-time estimate, not a measured
    real-world emergency response time and does not guarantee live traffic.
    """
    latitude = event.get("latitude")
    longitude = event.get("longitude")

    if not _valid_coordinates(latitude, longitude):
        return event

    latitude = float(latitude)
    longitude = float(longitude)

    cache_key = (
        round(latitude, 3),
        round(longitude, 3)
    )

    if cache_key in ROUTE_CACHE:
        event.update(ROUTE_CACHE[cache_key])
        return event

    facilities = _get_emergency_facilities(
        latitude,
        longitude
    )

    if not facilities:
        event["response_center_name"] = "No mapped response center found"
        event["response_center_type"] = "Unavailable"
        event["response_routing_source"] = "OpenStreetMap + OSRM"
        event["route_status"] = "NO_ROUTE_CENTER"
        return event

    # OSRM table calculates the fastest-route duration/distance
    # from the disaster to all candidate facilities in one request.
    coordinates = [
        f"{longitude},{latitude}"
    ]

    for facility in facilities:
        coordinates.append(
            f"{facility['longitude']},{facility['latitude']}"
        )

    coordinate_string = ";".join(coordinates)

    url = (
        f"{OSRM_TABLE_URL}/"
        f"{coordinate_string}"
    )

    try:
        response = requests.get(
            url,
            params={
                "sources": "0",
                "destinations":
                    ";".join(
                        str(i)
                        for i in range(1, len(coordinates))
                    ),
                "annotations": "duration,distance"
            },
            timeout=30
        )
        response.raise_for_status()
        data = response.json()

        durations = data.get("durations", [[]])[0]
        distances = data.get("distances", [[]])[0]

        candidates = []

        for index, duration in enumerate(durations):
            distance = (
                distances[index]
                if index < len(distances)
                else None
            )

            if duration is None or distance is None:
                continue

            facility = facilities[index]

            candidates.append({
                "facility": facility,
                "duration_seconds": float(duration),
                "distance_meters": float(distance)
            })

        if not candidates:
            raise RuntimeError("No road route found")

        best = min(
            candidates,
            key=lambda item: item["duration_seconds"]
        )

        facility = best["facility"]

        route_data = {
            "response_center_name":
                facility["name"],
            "response_center_type":
                facility["type"],
            "response_center_latitude":
                facility["latitude"],
            "response_center_longitude":
                facility["longitude"],

            # Keep the existing frontend field names.
            "estimated_travel_distance_km":
                round(
                    best["distance_meters"] / 1000,
                    2
                ),

            "estimated_response_time_minutes":
                round(
                    best["duration_seconds"] / 60,
                    2
                ),

            "response_routing_source":
                "OSRM road routing + OpenStreetMap",

            "route_status":
                "ROUTE_CALCULATED"
        }

        ROUTE_CACHE[cache_key] = route_data
        event.update(route_data)

    except Exception as error:
        event["response_center_name"] = (
            "Route unavailable"
        )
        event["response_center_type"] = "Unavailable"
        event["response_routing_source"] = (
            "OSRM road routing + OpenStreetMap"
        )
        event["route_status"] = "ROUTE_ERROR"
        event["route_error"] = str(error)

    return event


'''

(SERVICES / "routing_service.py").write_text(
    routing_code
)

print("routing_service.py created")


In [ ]:

import os
import time
import requests

# -------------------------------------------------------------------
# RESOURCE AVAILABILITY
# -------------------------------------------------------------------
# This service supports an optional verified operational JSON endpoint.
#
# If RESOURCE_STATUS_URL is configured, it should return:
# {
#   "rescue_teams": 8,
#   "medical_units": 12,
#   "relief_units": 20,
#   "source": "Your verified operational system",
#   "last_verified": "2026-09-28T10:00:00+05:30"
# }
#
# Without that endpoint, the project uses the configured demo inventory.
# It MUST NOT be described as real-world live availability.
# -------------------------------------------------------------------

RESOURCE_INVENTORY = {
    "rescue_teams": 3,
    "medical_units": 5,
    "relief_units": 5,
    "source": "Demo configuration - replace with verified inventory",
    "last_verified": None,
    "live": False,
}

_RESOURCE_CACHE = {
    "data": None,
    "fetched_at": 0.0,
}

RESOURCE_REFRESH_SECONDS = 30


def _clean_count(value, default=0):
    try:
        return max(0, int(value))
    except (TypeError, ValueError):
        return default


def _fetch_remote_inventory():
    url = os.getenv("RESOURCE_STATUS_URL", "").strip()

    if not url:
        return None

    response = requests.get(
        url,
        timeout=10,
        headers={
            "Accept": "application/json",
            "User-Agent": "QuantumDisasterResponse/1.0",
        },
    )
    response.raise_for_status()

    data = response.json()

    return {
        "rescue_teams": _clean_count(data.get("rescue_teams")),
        "medical_units": _clean_count(data.get("medical_units")),
        "relief_units": _clean_count(data.get("relief_units")),
        "source": data.get("source") or "Configured operational resource API",
        "last_verified": data.get("last_verified"),
        "live": bool(data.get("live", True)),
    }


def get_resource_inventory():
    now = time.time()

    if (
        _RESOURCE_CACHE["data"] is not None
        and now - _RESOURCE_CACHE["fetched_at"] < RESOURCE_REFRESH_SECONDS
    ):
        return _RESOURCE_CACHE["data"].copy()

    try:
        remote = _fetch_remote_inventory()

        if remote is not None:
            _RESOURCE_CACHE["data"] = remote
            _RESOURCE_CACHE["fetched_at"] = now
            return remote.copy()

    except Exception as error:
        print("Resource inventory refresh failed:", error)

    fallback = RESOURCE_INVENTORY.copy()
    _RESOURCE_CACHE["data"] = fallback
    _RESOURCE_CACHE["fetched_at"] = now
    return fallback


(SERVICES / "resource_service.py").write_text(
    resource_code,
    encoding="utf-8"
)

print("resource_service.py created")


In [ ]:
main_code = r'''

import asyncio

from contextlib import asynccontextmanager

from concurrent.futures import ThreadPoolExecutor

from fastapi import FastAPI

from fastapi.middleware.cors import CORSMiddleware

from global_land_mask import globe

from services.usgs_service import (
    get_earthquakes
)

from services.eonet_service import (
    get_eonet_events
)

from services.feature_service import (
    add_features
)

from services.ml_service import (
    predict_priority
)

from services.quantum_service import (
    quantum_allocate
)

from services.location_service import (
    get_location_name
)

from services.routing_service import (
    calculate_actual_route_to_response_center
)

from services.resource_service import (
    get_resource_inventory
)


latest_disaster_data = []

last_update_time = None

ocean_events_removed = 0


executor = ThreadPoolExecutor(
    max_workers=5
)


def is_land_event(event):

    latitude = event.get(
        "latitude"
    )

    longitude = event.get(
        "longitude"
    )


    if latitude is None or longitude is None:

        return False


    try:

        latitude = float(latitude)

        longitude = float(longitude)

    except (
        TypeError,
        ValueError
    ):

        return False


    if not (
        -90 <= latitude <= 90
    ):

        return False


    if not (
        -180 <= longitude <= 180
    ):

        return False


    return bool(

        globe.is_land(

            latitude,

            longitude

        )

    )


def process_event(event):

    event = add_features(
        event
    )

    event = predict_priority(
        event
    )


    event["location_name"] = (
        get_location_name(
            event.get(
                "latitude"
            ),
            event.get(
                "longitude"
            )
        )
    )


    return event


async def update_disaster_data():

    global latest_disaster_data

    global last_update_time

    global ocean_events_removed


    try:

        print(
            "Fetching live disaster data..."
        )


        earthquakes = (
            await asyncio.get_event_loop()
            .run_in_executor(
                executor,
                get_earthquakes
            )
        )


        eonet_events = (
            await asyncio.get_event_loop()
            .run_in_executor(
                executor,
                get_eonet_events
            )
        )


        combined_data = (

            earthquakes

            + eonet_events

        )


        land_events = [

            event

            for event in combined_data

            if is_land_event(
                event
            )

        ]


        ocean_events_removed = (

            len(combined_data)

            - len(land_events)

        )


        print(
            f"Total fetched: "
            f"{len(combined_data)}"
        )


        print(
            f"Land events: "
            f"{len(land_events)}"
        )


        print(
            f"Ocean events removed: "
            f"{ocean_events_removed}"
        )


        processed_data = []


        for event in land_events:

            try:

                processed_event = (
                    await asyncio.get_event_loop()
                    .run_in_executor(
                        executor,
                        process_event,
                        event
                    )
                )

                processed_data.append(
                    processed_event
                )

            except Exception as error:

                print(
                    "Event processing error:",
                    error
                )


        processed_data.sort(

            key=lambda x:
                x.get(
                    "priority_score",
                    0
                ),

            reverse=True

        )


        latest_disaster_data = (
            processed_data
        )


        from datetime import datetime

        last_update_time = (
            datetime.utcnow()
            .isoformat()
        )


        print(
            f"Update successful: "
            f"{len(processed_data)} disasters"
        )


    except Exception as error:

        print(
            "Update failed:",
            error
        )


async def background_updater():

    while True:

        await update_disaster_data()

        await asyncio.sleep(
            60
        )


@asynccontextmanager
async def lifespan(app):

    task = asyncio.create_task(
        background_updater()
    )

    yield

    task.cancel()


app = FastAPI(
    title="Quantum Disaster Response API",
    lifespan=lifespan
)


app.add_middleware(

    CORSMiddleware,

    allow_origins=[
        "*"
    ],

    allow_credentials=True,

    allow_methods=[
        "*"
    ],

    allow_headers=[
        "*"
    ]
)


@app.get("/")
def root():

    return {

        "message":
            "Quantum Disaster Response API",

        "status":
            "running",

        "disasters":
            len(
                latest_disaster_data
            )
    }


@app.get("/api/disasters")
def disasters():

    return {

        "count":
            len(
                latest_disaster_data
            ),

        "disasters":
            latest_disaster_data,

        "land_only":
            True,

        "ocean_events_removed":
            ocean_events_removed,

        "last_updated":
            last_update_time
    }


@app.get("/api/status")
def status():

    return {

        "status":
            "running",

        "disasters":
            len(
                latest_disaster_data
            ),

        "land_only":
            True,

        "ocean_events_removed":
            ocean_events_removed,

        "last_updated":
            last_update_time
    }


@app.get("/api/resources")
def resources():
    inventory = get_resource_inventory()

    return {
        "rescue_teams": inventory.get("rescue_teams", 0),
        "medical_units": inventory.get("medical_units", 0),
        "relief_units": inventory.get("relief_units", 0),
        "source": inventory.get("source"),
        "last_verified": inventory.get("last_verified"),
        "live": bool(inventory.get("live", False)),
    }


@app.get("/api/optimization")
def optimization(
    teams: int = None,
    medical: int = None,
    relief: int = None,
    available_teams: int = None,
    available_medical_units: int = None,
    available_relief_units: int = None
):

    inventory = get_resource_inventory()

    if available_teams is None:
        available_teams = (
            teams
            if teams is not None
            else inventory["rescue_teams"]
        )

    if available_medical_units is None:
        available_medical_units = (
            medical
            if medical is not None
            else inventory["medical_units"]
        )

    if available_relief_units is None:
        available_relief_units = (
            relief
            if relief is not None
            else inventory["relief_units"]
        )

    available_teams = max(
        1,
        min(100, int(available_teams))
    )

    available_medical_units = max(
        1,
        min(100, int(available_medical_units))
    )

    available_relief_units = max(
        1,
        min(100, int(available_relief_units))
    )


    if not latest_disaster_data:

        return {

            "selected": [],

            "objective": 0,

            "teams_used": 0,

            "medical_units_used": 0,

            "relief_units_used": 0,

            "available_teams":
                available_teams,

            "available_medical_units":
                available_medical_units,

            "available_relief_units":
                available_relief_units,

            "candidate_count": 0,

            "runtime_seconds": 0,

            "method": "QAOA",

            "status": "NO_DATA",
            "resource_inventory_source":
                inventory.get("source"),
            "resource_inventory_last_verified":
                inventory.get("last_verified")
        }


    result = quantum_allocate(

        latest_disaster_data,

        available_teams=
            available_teams,

        available_medical_units=
            available_medical_units,

        available_relief_units=
            available_relief_units

    )

    result["resource_inventory_source"] = (
        inventory.get("source")
    )
    result["resource_inventory_last_verified"] = (
        inventory.get("last_verified")
    )

    return result
'''

(BACKEND / "main.py").write_text(
    main_code
)

print("main.py created")

In [ ]:

print("""
UPDATED PROJECT BEHAVIOR

1. Disaster coordinates come from USGS/NASA EONET.
2. OpenStreetMap is queried for nearby mapped hospitals,
   fire stations, and disaster-response facilities.
3. OSRM calculates road-network distance and route duration.
4. The nearest/fastest response facility is selected.
5. The existing frontend fields
   estimated_travel_distance_km and
   estimated_response_time_minutes are now populated
   from the road-routing result for the top 10 disasters.
6. These are route-based travel estimates, not measured
   emergency response times and not guaranteed live-traffic times.
7. Resource counts remain a separate operational inventory.
   Replace RESOURCE_INVENTORY values with verified real counts.
""")


In [ ]:
for file in [
    BACKEND / "main.py",
    SERVICES / "usgs_service.py",
    SERVICES / "eonet_service.py",
    SERVICES / "feature_service.py",
    SERVICES / "ml_service.py",
    SERVICES / "quantum_service.py",
    SERVICES / "location_service.py"
]:

    print(
        "✓",
        file
    )

In [ ]:
import py_compile

files_to_check = [
    BACKEND / "main.py",
    SERVICES / "usgs_service.py",
    SERVICES / "eonet_service.py",
    SERVICES / "feature_service.py",
    SERVICES / "ml_service.py",
    SERVICES / "quantum_service.py",
    SERVICES / "location_service.py",
    SERVICES / "routing_service.py",
    SERVICES / "resource_service.py"
]

for file in files_to_check:

    py_compile.compile(
        str(file),
        doraise=True
    )

    print(
        "✓ Syntax OK:",
        file.name
    )

## Final backend configuration
This section replaces the repeated/diagnostic backend cells from the original notebook. Run All starts only one backend process and never kills it with `pkill`.

In [ ]:
from pathlib import Path

path = Path(
    "/content/quantum-disaster-response/backend/services/location_service.py"
)

code = '''
CACHE = {}

DISASTER_PREFIXES = [
    "Wildfire ",
    "Prescribed Fire ",
    "Tropical Cyclone ",
    "Flood ",
    "Landslide ",
    "Volcano ",
    "Severe Storms ",
    "Hurricane ",
    "Tornado "
]


def get_location_name(
    latitude=None,
    longitude=None,
    title=""
):

    # --------------------------------------------------
    # Extract Area, Place and State from title
    # --------------------------------------------------

    if title:

        parts = [
            part.strip()
            for part in title.split(",")
            if part.strip()
        ]

        if len(parts) >= 3:

            area = parts[-3]
            place = parts[-2]
            state = parts[-1]

            # Remove disaster type from Area
            for prefix in DISASTER_PREFIXES:

                if area.lower().startswith(
                    prefix.lower()
                ):

                    area = area[
                        len(prefix):
                    ].strip()

                    break

            if area:

                return (
                    f"{area}, "
                    f"{place}, "
                    f"{state}"
                )

        if len(parts) == 2:

            place = parts[-2]
            state = parts[-1]

            return (
                f"{place}, "
                f"{state}"
            )

    # --------------------------------------------------
    # Coordinate fallback
    # --------------------------------------------------

    if latitude is None or longitude is None:

        return "Location unavailable"

    try:

        latitude = float(latitude)
        longitude = float(longitude)

    except (TypeError, ValueError):

        return "Location unavailable"

    return (
        f"Coordinates: "
        f"{latitude:.3f}, "
        f"{longitude:.3f}"
    )
'''

path.write_text(
    code,
    encoding="utf-8"
)

print("✅ location_service.py updated")

In [ ]:
from pathlib import Path

path = Path("/content/quantum-disaster-response/backend/main.py")
text = path.read_text(encoding="utf-8")

# Make process_event fast: do not reverse-geocode every live event.
start = text.index("def process_event(event):")
end = text.index("\n\nasync def update_disaster_data():", start)

new_function = '''def process_event(event):
    event = add_features(event)
    event = predict_priority(event)

    # Location and road routing are added after sorting,
    # only for the top-priority events.
    event["location_name"] = "Location pending"
    event["route_status"] = "PENDING"

    return event
'''

text = text[:start] + new_function + text[end:]

# Add location lookup for the top 10 events after sorting.
marker = '''        latest_disaster_data = (
            processed_data
        )
'''

replacement = '''        # Add location + real road routing only to the top 10
        # priority disasters to keep live refresh practical.
        top_events = processed_data[:10]

        route_jobs = [
            asyncio.get_event_loop().run_in_executor(
                executor,
                calculate_actual_route_to_response_center,
                event
            )
            for event in top_events
        ]

        if route_jobs:
            await asyncio.gather(*route_jobs)

        for event in top_events:
            try:
                event["location_name"] = get_location_name(
                    event.get("latitude"),
                    event.get("longitude"),
                    event.get("title", "")
                )
            except Exception:
                event["location_name"] = "Location unavailable"

        latest_disaster_data = (
            processed_data
        )
'''

if marker not in text:
    raise RuntimeError("Could not find latest_disaster_data section in main.py.")

text = text.replace(marker, replacement, 1)
path.write_text(text, encoding="utf-8")

print("✅ main.py finalized")
print("✅ Location lookup limited to top 10 priority events")


In [ ]:
import py_compile

files_to_check = [
    BACKEND / "main.py",
    SERVICES / "usgs_service.py",
    SERVICES / "eonet_service.py",
    SERVICES / "feature_service.py",
    SERVICES / "ml_service.py",
    SERVICES / "quantum_service.py",
    SERVICES / "location_service.py",
    SERVICES / "routing_service.py",
    SERVICES / "resource_service.py"
]

for file in files_to_check:
    py_compile.compile(str(file), doraise=True)
    print("✓ Syntax OK:", file.name)

print("✅ Backend syntax check completed.")


In [ ]:
import subprocess
import time
import requests
from pathlib import Path

BACKEND = Path("/content/quantum-disaster-response/backend")
LOG_PATH = Path("/content/backend.log")

print("Checking FastAPI backend...")

backend_running = False

try:
    response = requests.get(
        "http://127.0.0.1:8000/api/status",
        timeout=3
    )
    if response.status_code == 200:
        backend_running = True
        print("✅ Backend is already running.")
        print(response.json())
except requests.exceptions.RequestException:
    print("Backend is not running.")

if not backend_running:
    print("\nStarting FastAPI backend...")

    log_file = open(LOG_PATH, "a")

    backend_process = subprocess.Popen(
        [
            "python",
            "-m",
            "uvicorn",
            "main:app",
            "--host",
            "0.0.0.0",
            "--port",
            "8000"
        ],
        cwd=str(BACKEND),
        stdout=log_file,
        stderr=subprocess.STDOUT,
        start_new_session=True
    )

    print("Backend process started.")
    print("PID:", backend_process.pid)

    for attempt in range(30):
        try:
            response = requests.get(
                "http://127.0.0.1:8000/api/status",
                timeout=3
            )

            if response.status_code == 200:
                backend_running = True
                print("\n✅ FastAPI backend is ready.")
                print(response.json())
                break

        except requests.exceptions.RequestException:
            pass

        print(f"Waiting for FastAPI... {attempt + 1}/30")
        time.sleep(2)

if not backend_running:
    print("\n❌ Backend failed to start.")
    if LOG_PATH.exists():
        print("\n--- Last backend log ---")
        print("".join(LOG_PATH.read_text(errors="ignore").splitlines(True)[-40:]))
    raise RuntimeError("FastAPI backend could not be started.")


In [ ]:
import requests
import time

print("Waiting for live disaster data...")

data_ready = False

for attempt in range(60):
    try:
        response = requests.get(
            "http://127.0.0.1:8000/api/status",
            timeout=5
        )
        response.raise_for_status()

        status = response.json()
        disasters = status.get("disasters", 0)
        last_updated = status.get("last_updated")

        if disasters > 0 and last_updated:
            print("\n✅ Live disaster data is ready.")
            print("Disasters loaded:", disasters)
            print("Last update:", last_updated)
            data_ready = True
            break

        print(f"Waiting for data... {attempt + 1}/60")

    except requests.exceptions.RequestException:
        print(f"Backend not ready... {attempt + 1}/60")

    time.sleep(2)

if not data_ready:
    print("\n❌ Live disaster data did not initialize within the wait period.")
    if Path("/content/backend.log").exists():
        print("\n--- Last backend log ---")
        print("".join(Path("/content/backend.log").read_text(errors="ignore").splitlines(True)[-50:]))
    raise RuntimeError("Live disaster data was not loaded.")


In [ ]:
import requests

response = requests.get(
    "http://127.0.0.1:8000/api/disasters",
    timeout=60
)
response.raise_for_status()

data = response.json()

print("Backend status:", response.status_code)
print("Total disasters:", data.get("count"))
print("Land only:", data.get("land_only"))
print("Ocean events removed:", data.get("ocean_events_removed"))
print("Last updated:", data.get("last_updated"))


In [ ]:
import requests

response = requests.get(
    "http://127.0.0.1:8000/api/optimization",
    params={"teams": 3, "medical": 5, "relief": 5},
    timeout=180
)
response.raise_for_status()

result = response.json()

print("Quantum method:", result.get("method"))
print("Status:", result.get("status"))
print("Objective:", result.get("objective"))
print("Rescue teams:", result.get("teams_used"), "/", result.get("available_teams"))
print("Medical units:", result.get("medical_units_used"), "/", result.get("available_medical_units"))
print("Relief units:", result.get("relief_units_used"), "/", result.get("available_relief_units"))


print("Resource inventory source:", result.get("resource_inventory_source"))
print("Selected response centers:")
for item in result.get("selected", []):
    print(
        " -",
        item.get("response_center_name"),
        "|",
        item.get("response_center_type"),
        "|",
        item.get("estimated_travel_distance_km"),
        "km |",
        item.get("estimated_response_time_minutes"),
        "min"
    )


## Final frontend setup
The working dashboard is preserved. The destructive frontend recreation cells from the original notebook are removed.

In [ ]:
from pathlib import Path
import subprocess

FRONTEND = Path("/content/frontend")

if not (FRONTEND / "package.json").exists():
    print("Creating React/Vite frontend...")
    result = subprocess.run(
        [
            "npm", "create", "vite@latest", "frontend",
            "--", "--template", "react"
        ],
        cwd="/content",
        capture_output=True,
        text=True,
        timeout=120
    )
    print(result.stdout)
    print(result.stderr)
else:
    print("✅ Existing frontend found. Keeping the working dashboard.")

result = subprocess.run(
    [
        "npm", "install",
        "axios", "leaflet", "react-leaflet", "recharts"
    ],
    cwd=str(FRONTEND),
    capture_output=True,
    text=True,
    timeout=180
)

print(result.stdout)
print(result.stderr)

if result.returncode != 0:
    raise RuntimeError("Frontend npm install failed.")

print("✅ Frontend dependencies ready.")


In [ ]:
from pathlib import Path

vite_config = r'''
import { defineConfig } from "vite";
import react from "@vitejs/plugin-react";

export default defineConfig({
  plugins: [react()],

  server: {
    host: "0.0.0.0",
    port: 5173,

    allowedHosts: true,

    proxy: {
      "/api": {
        target: "http://127.0.0.1:8000",
        changeOrigin: true,
        secure: false
      }
    }
  }
});
'''

Path("/content/frontend/vite.config.js").write_text(
    vite_config,
    encoding="utf-8"
)

print("✅ Vite proxy configured")

In [ ]:
from pathlib import Path

app_jsx = r'''
import { useEffect, useMemo, useState } from "react";
import axios from "axios";
import {
  MapContainer,
  TileLayer,
  CircleMarker,
  Popup,
  useMap
} from "react-leaflet";
import "leaflet/dist/leaflet.css";
import "./App.css";

const API_BASE = "";

const api = axios.create({
  baseURL: API_BASE,
  timeout: 120000
});

const PRIORITY_COLORS = {
  CRITICAL: "#ef4444",
  HIGH: "#f97316",
  MEDIUM: "#eab308",
  LOW: "#22c55e"
};

function FitMap({ disasters }) {
  const map = useMap();

  useEffect(() => {
    const valid = disasters.filter(
      d =>
        Number.isFinite(Number(d.latitude)) &&
        Number.isFinite(Number(d.longitude))
    );

    if (!valid.length) return;

    const bounds = valid.map(d => [
      Number(d.latitude),
      Number(d.longitude)
    ]);

    map.fitBounds(bounds, {
      padding: [35, 35],
      maxZoom: 4
    });
  }, [disasters, map]);

  return null;
}

function getPriorityColor(level) {
  return PRIORITY_COLORS[level] || PRIORITY_COLORS.LOW;
}

function formatNumber(value, digits = 0) {
  if (value === null || value === undefined || value === "") {
    return "N/A";
  }

  const number = Number(value);

  if (!Number.isFinite(number)) {
    return String(value);
  }

  return number.toFixed(digits);
}

function App() {
  const [disasters, setDisasters] = useState([]);
  const [quantumResult, setQuantumResult] = useState(null);

  const [loading, setLoading] = useState(true);
  const [quantumLoading, setQuantumLoading] = useState(false);
  const [scenarioLoading, setScenarioLoading] = useState(false);

  const [error, setError] = useState("");
  const [lastUpdated, setLastUpdated] = useState(null);
  const [resourceInventory, setResourceInventory] = useState(null);

  const [search, setSearch] = useState("");
  const [sourceFilter, setSourceFilter] = useState("ALL");
  const [priorityFilter, setPriorityFilter] = useState("ALL");
  const [typeFilter, setTypeFilter] = useState("ALL");

  const [scenarioTeams, setScenarioTeams] = useState(3);
  const [scenarioMedical, setScenarioMedical] = useState(5);
  const [scenarioRelief, setScenarioRelief] = useState(5);
  const [scenarioResult, setScenarioResult] = useState(null);

  const fetchDisasters = async (firstLoad = false) => {
    try {
      if (firstLoad) {
        setLoading(true);
      }

      const response = await api.get("/api/disasters");

      const list = response.data?.disasters || [];

      setDisasters(list);
      setLastUpdated(response.data?.last_updated || null);
      setError("");
    } catch (err) {
      console.error("Disaster API error:", err);

      if (firstLoad) {
        setError(
          "Unable to connect to the disaster backend."
        );
      }
    } finally {
      if (firstLoad) {
        setLoading(false);
      }
    }
  };

  const fetchResourceInventory = async () => {
    try {
      const response = await api.get("/api/resources", {
        timeout: 15000
      });
      setResourceInventory(response.data || null);
    } catch (err) {
      console.error("Resource inventory error:", err);
    }
  };

  const runQuantumOptimization = async (
    teams = 3,
    medical = 5,
    relief = 5,
    scenario = false
  ) => {
    try {
      if (scenario) {
        setScenarioLoading(true);
      } else {
        setQuantumLoading(true);
      }

      const url = scenario
        ? `/api/optimization?available_teams=${teams}&available_medical_units=${medical}&available_relief_units=${relief}`
        : "/api/optimization";

      const response = await api.get(url, {
        timeout: 120000
      });

      if (scenario) {
        setScenarioResult(response.data);
      } else {
        setQuantumResult(response.data);
      }

      setError("");
    } catch (err) {
      console.error("Optimization error:", err);

      setError(
        "Quantum optimization could not be completed."
      );
    } finally {
      if (scenario) {
        setScenarioLoading(false);
      } else {
        setQuantumLoading(false);
      }
    }
  };

  useEffect(() => {
    fetchDisasters(true);
    fetchResourceInventory();

    const disasterInterval = setInterval(() => {
      fetchDisasters(false);
    }, 60000);

    const resourceInterval = setInterval(() => {
      fetchResourceInventory();
    }, 30000);

    return () => {
      clearInterval(disasterInterval);
      clearInterval(resourceInterval);
    };
  }, []);

  const sources = useMemo(() => {
    return [
      ...new Set(
        disasters
          .map(d => d.source)
          .filter(Boolean)
      )
    ].sort();
  }, [disasters]);

  const types = useMemo(() => {
    return [
      ...new Set(
        disasters
          .map(d => d.type)
          .filter(Boolean)
      )
    ].sort();
  }, [disasters]);

  const filteredDisasters = useMemo(() => {
    const query = search.trim().toLowerCase();

    return disasters.filter(d => {
      const matchesSearch =
        !query ||
        String(d.title || "")
          .toLowerCase()
          .includes(query) ||
        String(d.location_name || "")
          .toLowerCase()
          .includes(query) ||
        String(d.place || "")
          .toLowerCase()
          .includes(query);

      const matchesSource =
        sourceFilter === "ALL" ||
        d.source === sourceFilter;

      const matchesPriority =
        priorityFilter === "ALL" ||
        d.priority_level === priorityFilter;

      const matchesType =
        typeFilter === "ALL" ||
        d.type === typeFilter;

      return (
        matchesSearch &&
        matchesSource &&
        matchesPriority &&
        matchesType
      );
    });
  }, [
    disasters,
    search,
    sourceFilter,
    priorityFilter,
    typeFilter
  ]);

  const stats = useMemo(() => {
    const critical = disasters.filter(
      d => d.priority_level === "CRITICAL"
    ).length;

    const high = disasters.filter(
      d => d.priority_level === "HIGH"
    ).length;

    const medium = disasters.filter(
      d => d.priority_level === "MEDIUM"
    ).length;

    const low = disasters.filter(
      d => d.priority_level === "LOW"
    ).length;

    return {
      total: disasters.length,
      critical,
      high,
      medium,
      low
    };
  }, [disasters]);

  const topDisasters = useMemo(() => {
    return [...disasters]
      .sort(
        (a, b) =>
          Number(b.priority_score || 0) -
          Number(a.priority_score || 0)
      )
      .slice(0, 10);
  }, [disasters]);

  const resourceResult =
    quantumResult || scenarioResult;

  const resetFilters = () => {
    setSearch("");
    setSourceFilter("ALL");
    setPriorityFilter("ALL");
    setTypeFilter("ALL");
  };

  const runScenario = () => {
    runQuantumOptimization(
      Number(scenarioTeams),
      Number(scenarioMedical),
      Number(scenarioRelief),
      true
    );
  };

  if (loading && disasters.length === 0 && !error) {
    return (
      <div className="loading-screen">
        <div className="loading-card">
          <div className="quantum-loader">⚛</div>
          <h1>Quantum Disaster Response</h1>
          <p>
            Loading live disaster intelligence...
          </p>
        </div>
      </div>
    );
  }

  return (
    <div className="app">

      {/* HEADER */}

      <header className="top-header">

        <div className="brand-area">

          <div className="brand-icon">
            ⚛
          </div>

          <div>
            <h1>
              Quantum Disaster Response
            </h1>

            <p>
              AI-powered disaster intelligence &
              quantum resource optimization
            </p>
          </div>

        </div>

        <div className="header-actions">

          <div className="live-badge">
            <span className="live-dot"></span>
            LIVE
          </div>

          <button
            className="quantum-main-button"
            onClick={() =>
              runQuantumOptimization()
            }
            disabled={quantumLoading}
          >
            ⚛{" "}
            {quantumLoading
              ? "Running QAOA..."
              : "Run Quantum Optimization"}
          </button>

        </div>

      </header>

      {/* ERROR */}

      {error && (
        <div className="error-banner">
          ⚠️ {error}
        </div>
      )}

      {/* STATISTICS */}

      <section className="stats-section">

        <div className="stat-card">

          <div className="stat-icon blue">
            🌐
          </div>

          <div>
            <span className="stat-label">
              ACTIVE DISASTERS
            </span>

            <strong className="stat-value">
              {stats.total}
            </strong>

            <small>
              Events matching filters
            </small>
          </div>

        </div>

        <div className="stat-card">

          <div className="stat-icon orange">
            🔥
          </div>

          <div>
            <span className="stat-label">
              HIGH PRIORITY
            </span>

            <strong className="stat-value">
              {stats.high}
            </strong>

            <small>
              High + critical events
            </small>
          </div>

        </div>

        <div className="stat-card">

          <div className="stat-icon red">
            🚨
          </div>

          <div>
            <span className="stat-label">
              CRITICAL
            </span>

            <strong className="stat-value">
              {stats.critical}
            </strong>

            <small>
              Immediate response
            </small>
          </div>

        </div>

        <div className="stat-card">

          <div className="stat-icon purple">
            ⚛
          </div>

          <div>
            <span className="stat-label">
              QUANTUM METHOD
            </span>

            <strong className="stat-method">
              QAOA
            </strong>

            <small>
              Quantum optimization
            </small>
          </div>

        </div>

      </section>

      {/* RESOURCE AVAILABILITY */}

      <section className="panel resource-status-panel">

        <div className="section-heading">
          <div>
            <h2>🛰 Operational Resource Availability</h2>
            <p>
              Current inventory used as the QAOA constraint
            </p>
          </div>

          <span className={`inventory-badge ${resourceInventory?.live ? "live" : "configured"}`}>
            {resourceInventory?.live ? "● LIVE SOURCE" : "● CONFIGURED SOURCE"}
          </span>
        </div>

        <div className="resource-status-grid">

          <div>
            <span>🚑 RESCUE TEAMS</span>
            <strong>{resourceInventory?.rescue_teams ?? "—"}</strong>
          </div>

          <div>
            <span>🏥 MEDICAL UNITS</span>
            <strong>{resourceInventory?.medical_units ?? "—"}</strong>
          </div>

          <div>
            <span>📦 RELIEF UNITS</span>
            <strong>{resourceInventory?.relief_units ?? "—"}</strong>
          </div>

        </div>

        <div className="resource-source">
          Source: {resourceInventory?.source || "Not available"}
          {" • "}
          Last verified: {resourceInventory?.last_verified || "Not provided"}
        </div>

      </section>

      {/* FILTERS */}

      <section className="panel filter-panel">

        <div className="panel-title">

          <div className="title-icon">
            🔎
          </div>

          <div>
            <h2>Disaster Filters</h2>
            <p>
              Search and filter live disaster events
            </p>
          </div>

        </div>

        <div className="filter-row">

          <div className="search-box">

            <span>⌕</span>

            <input
              type="text"
              placeholder="Search disaster or location..."
              value={search}
              onChange={e =>
                setSearch(e.target.value)
              }
            />

          </div>

          <select
            value={sourceFilter}
            onChange={e =>
              setSourceFilter(e.target.value)
            }
          >
            <option value="ALL">
              All Sources
            </option>

            {sources.map(source => (
              <option
                key={source}
                value={source}
              >
                {source}
              </option>
            ))}
          </select>

          <select
            value={priorityFilter}
            onChange={e =>
              setPriorityFilter(e.target.value)
            }
          >
            <option value="ALL">
              All Priorities
            </option>

            <option value="CRITICAL">
              Critical
            </option>

            <option value="HIGH">
              High
            </option>

            <option value="MEDIUM">
              Medium
            </option>

            <option value="LOW">
              Low
            </option>
          </select>

          <select
            value={typeFilter}
            onChange={e =>
              setTypeFilter(e.target.value)
            }
          >
            <option value="ALL">
              All Types
            </option>

            {types.map(type => (
              <option
                key={type}
                value={type}
              >
                {type}
              </option>
            ))}
          </select>

          <button
            className="reset-button"
            onClick={resetFilters}
          >
            Reset
          </button>

        </div>

        <div className="filter-footer">

          <strong>
            {filteredDisasters.length} results
          </strong>

          <span>
            Disaster Monitoring Active
          </span>

        </div>

      </section>

      {/* UPDATE STATUS */}

      <div className="monitoring-bar">

        <div>
          <span className="status-dot"></span>
          <strong>
            Disaster Monitoring Active
          </strong>
        </div>

        <span>
          Data updates every 60 seconds • Last update:{" "}
          {lastUpdated
            ? new Date(
                lastUpdated
              ).toLocaleTimeString()
            : "N/A"}
        </span>

      </div>

      {/* MAP */}

      <section className="panel map-panel">

        <div className="section-heading">

          <div>
            <h2>🌍 Live Disaster Map</h2>

            <p>
              Real-time disaster events from USGS
              and NASA EONET
            </p>
          </div>

          <div className="map-count">
            {filteredDisasters.length} mapped events
          </div>

        </div>

        <div className="map-wrapper">

          <MapContainer
            center={[20, 0]}
            zoom={2}
            scrollWheelZoom={true}
            className="disaster-map"
          >

            <TileLayer
              attribution='&copy; OpenStreetMap contributors'
              url="https://{s}.tile.openstreetmap.org/{z}/{x}/{y}.png"
            />

            <FitMap
              disasters={filteredDisasters}
            />

            {filteredDisasters.map(
              (disaster, index) => {

                const lat =
                  Number(disaster.latitude);

                const lon =
                  Number(disaster.longitude);

                if (
                  !Number.isFinite(lat) ||
                  !Number.isFinite(lon)
                ) {
                  return null;
                }

                const level =
                  disaster.priority_level ||
                  "LOW";

                const color =
                  getPriorityColor(level);

                const quantumSelected =
                  (quantumResult?.selected || [])
                    .some(
                      item =>
                        item.id &&
                        item.id === disaster.id
                    );

                return (
                  <CircleMarker
                    key={
                      disaster.id ||
                      `${disaster.source}-${index}`
                    }
                    center={[lat, lon]}
                    radius={
                      quantumSelected
                        ? 13
                        : level === "CRITICAL"
                        ? 10
                        : level === "HIGH"
                        ? 8
                        : 6
                    }
                    pathOptions={{
                      color: quantumSelected
                        ? "#7c3aed"
                        : color,
                      fillColor: quantumSelected
                        ? "#a855f7"
                        : color,
                      fillOpacity: 0.8,
                      weight: quantumSelected
                        ? 4
                        : 2
                    }}
                  >

                    <Popup>

                      <div className="map-popup">

                        <div className="popup-header">

                          <span
                            className="popup-priority"
                            style={{
                              background:
                                color
                            }}
                          >
                            {level}
                          </span>

                          <span>
                            {disaster.source}
                          </span>

                        </div>

                        <h3>
                          {disaster.title ||
                            disaster.type ||
                            "Disaster Event"}
                        </h3>

                        <div className="popup-location">
                          📍{" "}
                          <strong>
                            {disaster.location_name ||
                              "Location unavailable"}
                          </strong>
                        </div>

                        <div className="popup-grid">

                          <div>
                            <span>Type</span>
                            <strong>
                              {disaster.type ||
                                "Unknown"}
                            </strong>
                          </div>

                          <div>
                            <span>Priority</span>
                            <strong>
                              {formatNumber(
                                disaster.priority_score,
                                2
                              )}
                            </strong>
                          </div>

                          <div>
                            <span>Severity</span>
                            <strong>
                              {formatNumber(
                                disaster.severity,
                                0
                              )}
                            </strong>
                          </div>

                          <div>
                            <span>Response</span>
                            <strong>
                              {formatNumber(
                                disaster.estimated_response_time_minutes,
                                0
                              )}{" "}
                              min
                            </strong>
                          </div>

                        </div>

                        <div className="popup-details">

                          <p>
                            👥 People affected:{" "}
                            {formatNumber(
                              disaster.estimated_people_affected
                            )}
                          </p>

                          <p>
                            🏥 Medical need:{" "}
                            {formatNumber(
                              disaster.estimated_medical_need
                            )}
                          </p>

                          <p>
                            🚑 Rescue need:{" "}
                            {formatNumber(
                              disaster.estimated_rescue_need
                            )}
                          </p>

                          <p>
                            🛣️ Travel distance:{" "}
                            {formatNumber(
                              disaster.estimated_travel_distance_km
                            )}{" "}
                            km
                          </p>

                        </div>

                        {quantumSelected && (
                          <div className="quantum-selected">
                            ⚛ Quantum Selected
                          </div>
                        )}

                      </div>

                    </Popup>

                  </CircleMarker>
                );
              }
            )}

          </MapContainer>

          <div className="map-legend">

            <strong>Priority Level</strong>

            <div>
              <span className="legend-dot critical"></span>
              Critical
            </div>

            <div>
              <span className="legend-dot high"></span>
              High
            </div>

            <div>
              <span className="legend-dot medium"></span>
              Medium
            </div>

            <div>
              <span className="legend-dot low"></span>
              Low
            </div>

            <div>
              <span className="legend-dot quantum"></span>
              Quantum Selected
            </div>

          </div>

        </div>

      </section>

      {/* QUANTUM OPTIMIZATION */}

      <section className="panel quantum-panel">

        <div className="section-heading">

          <div>
            <h2>⚛ Quantum Optimization</h2>

            <p>
              QAOA-based rescue resource allocation
            </p>
          </div>

          {quantumResult && (
            <span className="success-badge">
              ● SUCCESS
            </span>
          )}

        </div>

        {!quantumResult ? (

          <div className="quantum-empty">

            <div className="large-quantum">
              ⚛
            </div>

            <h3>
              Quantum optimization ready
            </h3>

            <p>
              Run QAOA to optimize limited rescue,
              medical and relief resources.
            </p>

            <button
              className="quantum-main-button"
              onClick={() =>
                runQuantumOptimization()
              }
              disabled={quantumLoading}
            >
              ⚛ Run Quantum Optimization
            </button>

          </div>

        ) : (

          <>

            <div className="quantum-metrics">

              <div>
                <span>METHOD</span>
                <strong>
                  {quantumResult.method ||
                    "QAOA"}
                </strong>
              </div>

              <div>
                <span>RESCUE TEAMS</span>
                <strong>
                  {quantumResult.teams_used ??
                    0}
                  /
                  {quantumResult.available_teams ??
                    0}
                </strong>
              </div>

              <div>
                <span>MEDICAL UNITS</span>
                <strong>
                  {quantumResult.medical_units_used ??
                    0}
                  /
                  {quantumResult.available_medical_units ??
                    0}
                </strong>
              </div>

              <div>
                <span>RELIEF UNITS</span>
                <strong>
                  {quantumResult.relief_units_used ??
                    0}
                  /
                  {quantumResult.available_relief_units ??
                    0}
                </strong>
              </div>

              <div>
                <span>OBJECTIVE</span>
                <strong>
                  {formatNumber(
                    quantumResult.objective,
                    2
                  )}
                </strong>
              </div>

              <div>
                <span>QAOA RUNTIME</span>
                <strong>
                  {formatNumber(
                    quantumResult.runtime_seconds,
                    3
                  )}s
                </strong>
              </div>

            </div>

            <div className="resource-section">

              <h3>
                📊 Resource Utilization
              </h3>

              <ResourceBar
                label="Rescue Teams"
                used={
                  quantumResult.teams_used
                }
                total={
                  quantumResult.available_teams
                }
                icon="🚑"
              />

              <ResourceBar
                label="Medical Units"
                used={
                  quantumResult.medical_units_used
                }
                total={
                  quantumResult.available_medical_units
                }
                icon="🏥"
              />

              <ResourceBar
                label="Relief Units"
                used={
                  quantumResult.relief_units_used
                }
                total={
                  quantumResult.available_relief_units
                }
                icon="📦"
              />

            </div>

          </>

        )}

      </section>

      {/* SCENARIO SIMULATOR */}

      <section className="panel scenario-panel">

        <div className="section-heading">

          <div>
            <h2>🧪 Scenario Simulator</h2>

            <p>
              Adjust available resources to test
              different disaster-response scenarios.
            </p>
          </div>

        </div>

        <div className="scenario-controls">

          <div className="scenario-input">

            <label>
              🚑 Rescue Teams
            </label>

            <input
              type="number"
              min="1"
              max="10"
              value={scenarioTeams}
              onChange={e =>
                setScenarioTeams(
                  e.target.value
                )
              }
            />

          </div>

          <div className="scenario-input">

            <label>
              🏥 Medical Units
            </label>

            <input
              type="number"
              min="1"
              max="20"
              value={scenarioMedical}
              onChange={e =>
                setScenarioMedical(
                  e.target.value
                )
              }
            />

          </div>

          <div className="scenario-input">

            <label>
              📦 Relief Units
            </label>

            <input
              type="number"
              min="1"
              max="20"
              value={scenarioRelief}
              onChange={e =>
                setScenarioRelief(
                  e.target.value
                )
              }
            />

          </div>

          <button
            className="scenario-button"
            onClick={runScenario}
            disabled={scenarioLoading}
          >
            ⚛{" "}
            {scenarioLoading
              ? "Optimizing..."
              : "Run Scenario Optimization"}
          </button>

        </div>

        <div className="scenario-current">

          Current Scenario:{" "}
          <strong>
            {scenarioTeams} rescue teams
          </strong>{" "}
          •{" "}
          <strong>
            {scenarioMedical} medical units
          </strong>{" "}
          •{" "}
          <strong>
            {scenarioRelief} relief units
          </strong>

        </div>

        {scenarioResult && (

          <div className="scenario-result">

            <h3>
              📊 Scenario Impact
            </h3>

            <p>
              Resource allocation produced by the
              QAOA optimization for the selected
              scenario.
            </p>

            <div className="scenario-metrics">

              <div>
                <span>🚑 RESCUE TEAMS</span>
                <strong>
                  {scenarioResult.teams_used}/
                  {scenarioResult.available_teams}
                </strong>
              </div>

              <div>
                <span>🏥 MEDICAL UNITS</span>
                <strong>
                  {scenarioResult.medical_units_used}/
                  {scenarioResult.available_medical_units}
                </strong>
              </div>

              <div>
                <span>📦 RELIEF UNITS</span>
                <strong>
                  {scenarioResult.relief_units_used}/
                  {scenarioResult.available_relief_units}
                </strong>
              </div>

            </div>

            <div className="scenario-status">
              Optimization Status:{" "}
              <strong>
                {scenarioResult.status}
              </strong>{" "}
              • Method:{" "}
              <strong>
                {scenarioResult.method}
              </strong>{" "}
              • Objective:{" "}
              <strong>
                {formatNumber(
                  scenarioResult.objective,
                  2
                )}
              </strong>
            </div>

          </div>

        )}

      </section>

      {/* RECOMMENDATIONS */}

      <section className="panel recommendations-panel">

        <div className="section-heading">

          <div>
            <h2>
              🎯 Quantum Recommended Areas
            </h2>

            <p>
              Locations selected by the quantum
              optimization model
            </p>
          </div>

        </div>

        <div className="recommendation-list">

          {(resourceResult?.selected || []).length ===
          0 ? (

            <div className="no-recommendations">
              Run quantum optimization to generate
              recommended disaster-response areas.
            </div>

          ) : (

            (resourceResult?.selected || []).map(
              (item, index) => (

                <div
                  className="recommendation-card"
                  key={
                    item.id ||
                    `${index}-${item.title}`
                  }
                >

                  <div className="recommendation-number">
                    {index + 1}
                  </div>

                  <div className="recommendation-main">

                    <div className="recommendation-title">
                      {item.title ||
                        item.type ||
                        "Disaster"}
                    </div>

                    <span className="recommendation-type">
                      {item.type ||
                        "Unknown type"}
                    </span>

                    <div className="recommendation-location">
                      📍{" "}
                      {item.location_name ||
                        "Location unavailable"}
                    </div>

                    <div className="recommendation-resources">

                      Teams required:{" "}
                      {item.estimated_rescue_teams_needed ??
                        1}

                      {" • "}

                      Medical required:{" "}
                      {item.estimated_medical_units_needed ??
                        1}

                      {" • "}

                      Relief required:{" "}
                      {item.estimated_relief_units_needed ??
                        1}

                    </div>

                    <div className="recommendation-why">

                      <strong>
                        Why this disaster was selected:
                      </strong>{" "}

                      Priority score:{" "}
                      {formatNumber(
                        item.priority_score,
                        2
                      )}

                      {" • "}

                      Rescue teams:{" "}
                      {item.estimated_rescue_teams_needed ??
                        1}

                      {" • "}

                      Medical units:{" "}
                      {item.estimated_medical_units_needed ??
                        1}

                      {" • "}

                      Relief units:{" "}
                      {item.estimated_relief_units_needed ??
                        1}

                      {" • "}

                      Response time:{" "}
                      {formatNumber(
                        item.estimated_response_time_minutes
                      )}{" "}
                      min

                      {" • "}

                      Travel distance:{" "}
                      {formatNumber(
                        item.estimated_travel_distance_km
                      )}{" "}
                      km

                    </div>

                  </div>

                  <div className="recommendation-score">

                    <span>
                      PRIORITY
                    </span>

                    <strong>
                      {formatNumber(
                        item.priority_score,
                        2
                      )}
                    </strong>

                  </div>

                </div>

              )
            )

          )}

        </div>

      </section>

      {/* PRIORITY DISTRIBUTION */}

      <section className="panel distribution-panel">

        <div className="section-heading">

          <div>
            <h2>
              📊 Priority Distribution
            </h2>

            <p>
              Machine Learning classification
            </p>
          </div>

        </div>

        <div className="distribution-grid">

          <DistributionCard
            label="Critical"
            value={stats.critical}
            color="critical"
          />

          <DistributionCard
            label="High"
            value={stats.high}
            color="high"
          />

          <DistributionCard
            label="Medium"
            value={stats.medium}
            color="medium"
          />

          <DistributionCard
            label="Low"
            value={stats.low}
            color="low"
          />

        </div>

      </section>

      {/* TOP DISASTERS */}

      <section className="panel table-panel">

        <div className="section-heading">

          <div>
            <h2>
              🚨 Top Priority Disasters
            </h2>

            <p>
              Highest priority events identified by
              Machine Learning
            </p>
          </div>

          <span className="table-badge">
            Top 10
          </span>

        </div>

        <div className="table-wrapper">

          <table>

            <thead>

              <tr>
                <th>#</th>
                <th>DISASTER</th>
                <th>LOCATION</th>
                <th>SOURCE</th>
                <th>TYPE</th>
                <th>PRIORITY</th>
                <th>LEVEL</th>
              </tr>

            </thead>

            <tbody>

              {topDisasters.map(
                (disaster, index) => {

                  const level =
                    disaster.priority_level ||
                    "LOW";

                  return (
                    <tr
                      key={
                        disaster.id ||
                        `${index}-${disaster.title}`
                      }
                    >

                      <td className="rank">
                        {index + 1}
                      </td>

                      <td className="disaster-name">
                        {disaster.title ||
                          disaster.type ||
                          "Unknown"}
                      </td>

                      <td>
                        {disaster.location_name ||
                          "Location unavailable"}
                      </td>

                      <td>
                        {disaster.source ||
                          "Unknown"}
                      </td>

                      <td>
                        {disaster.type ||
                          "Unknown"}
                      </td>

                      <td className="priority-value">
                        {formatNumber(
                          disaster.priority_score,
                          2
                        )}
                      </td>

                      <td>

                        <span
                          className={`level-badge ${level.toLowerCase()}`}
                        >
                          {level}
                        </span>

                      </td>

                    </tr>
                  );
                }
              )}

            </tbody>

          </table>

        </div>

      </section>

      {/* FOOTER */}

      <footer>

        <div className="footer-logo">
          ⚛
        </div>

        <h2>
          Quantum Disaster Response System
        </h2>

        <p>
          USGS • NASA EONET • Machine Learning • QAOA
        </p>
        <small>
          Disaster data refreshes every 60 seconds. Resource inventory refreshes every 30 seconds when a verified live source is configured.
        </small>

      </footer>

    </div>
  );
}

function ResourceBar({
  label,
  used,
  total,
  icon
}) {
  const safeUsed = Number(used || 0);
  const safeTotal = Number(total || 0);

  const percentage =
    safeTotal > 0
      ? Math.min(
          100,
          Math.round(
            (safeUsed / safeTotal) * 100
          )
        )
      : 0;

  return (
    <div className="resource-row">

      <div className="resource-label">
        <span>
          {icon} {label}
        </span>

        <strong>
          {safeUsed} / {safeTotal}
        </strong>
      </div>

      <div className="progress-track">
        <div
          className="progress-fill"
          style={{
            width: `${percentage}%`
          }}
        ></div>
      </div>

      <small>
        {percentage}% utilized
      </small>

    </div>
  );
}

function DistributionCard({
  label,
  value,
  color
}) {
  return (
    <div className="distribution-card">

      <div
        className={`distribution-dot ${color}`}
      ></div>

      <div>
        <span>{label}</span>
        <strong>{value}</strong>
      </div>

    </div>
  );
}

export default App;
'''

Path("/content/frontend/src/App.jsx").write_text(
    app_jsx,
    encoding="utf-8"
)

print("✅ Complete App.jsx replaced")
print(
    "File size:",
    Path("/content/frontend/src/App.jsx").stat().st_size,
    "bytes"
)

In [ ]:
from pathlib import Path

app_css = r'''
:root {
  font-family:
    Inter,
    ui-sans-serif,
    system-ui,
    -apple-system,
    BlinkMacSystemFont,
    "Segoe UI",
    sans-serif;

  color: #172033;
  background: #f5f7fb;
  font-synthesis: none;
  text-rendering: optimizeLegibility;
}

* {
  box-sizing: border-box;
}

html {
  scroll-behavior: smooth;
}

body {
  margin: 0;
  min-width: 320px;
  min-height: 100vh;
  background: #f5f7fb;
}

button,
input,
select {
  font: inherit;
}

button {
  cursor: pointer;
}

button:disabled {
  cursor: not-allowed;
  opacity: 0.65;
}

.app {
  min-height: 100vh;
}

/* =========================================
   HEADER
========================================= */

.top-header {
  min-height: 92px;
  padding: 20px 42px;

  display: flex;
  justify-content: space-between;
  align-items: center;
  gap: 25px;

  color: white;

  background:
    radial-gradient(
      circle at 15% 10%,
      rgba(99, 102, 241, 0.25),
      transparent 30%
    ),
    linear-gradient(
      135deg,
      #111827,
      #172554 55%,
      #312e81
    );

  border-bottom: 1px solid rgba(255,255,255,0.08);
}

.brand-area {
  display: flex;
  align-items: center;
  gap: 16px;
}

.brand-icon {
  width: 48px;
  height: 48px;

  display: flex;
  justify-content: center;
  align-items: center;

  border-radius: 14px;

  background: rgba(255,255,255,0.12);

  font-size: 29px;

  box-shadow:
    0 8px 25px rgba(0,0,0,0.18);
}

.brand-area h1 {
  margin: 0;

  font-size: 25px;
  line-height: 1.2;
  font-weight: 750;
  letter-spacing: -0.5px;
}

.brand-area p {
  margin: 5px 0 0;

  color: rgba(255,255,255,0.72);

  font-size: 13px;
}

.header-actions {
  display: flex;
  align-items: center;
  gap: 12px;
}

.live-badge {
  display: flex;
  align-items: center;
  gap: 7px;

  padding: 9px 13px;

  border-radius: 20px;

  background: rgba(34,197,94,0.12);
  border: 1px solid rgba(74,222,128,0.3);

  color: #bbf7d0;

  font-size: 11px;
  font-weight: 800;
  letter-spacing: 0.5px;
}

.live-dot {
  width: 7px;
  height: 7px;

  border-radius: 50%;

  background: #22c55e;

  box-shadow:
    0 0 0 4px rgba(34,197,94,0.12);
}

.quantum-main-button {
  border: 0;

  padding: 11px 17px;

  border-radius: 9px;

  color: white;

  background:
    linear-gradient(
      135deg,
      #7c3aed,
      #4f46e5
    );

  font-size: 12px;
  font-weight: 750;

  box-shadow:
    0 6px 20px rgba(79,70,229,0.25);

  transition:
    transform 0.15s ease,
    box-shadow 0.15s ease;
}

.quantum-main-button:hover {
  transform: translateY(-1px);

  box-shadow:
    0 9px 25px rgba(79,70,229,0.32);
}

/* =========================================
   ERROR
========================================= */

.error-banner {
  margin: 18px 42px 0;

  padding: 13px 16px;

  border-radius: 10px;

  background: #fef2f2;
  border: 1px solid #fecaca;

  color: #991b1b;

  font-size: 13px;
  font-weight: 600;
}

/* =========================================
   STATISTICS
========================================= */

.stats-section {
  padding: 25px 42px 10px;

  display: grid;

  grid-template-columns:
    repeat(4, minmax(0, 1fr));

  gap: 15px;
}

.stat-card {
  min-height: 125px;

  display: flex;
  align-items: center;

  gap: 15px;

  padding: 20px;

  background: white;

  border: 1px solid #e8ebf2;

  border-radius: 14px;

  box-shadow:
    0 5px 20px rgba(15,23,42,0.045);
}

.stat-icon {
  width: 48px;
  height: 48px;

  flex-shrink: 0;

  display: flex;
  justify-content: center;
  align-items: center;

  border-radius: 12px;

  font-size: 23px;
}

.stat-icon.blue {
  background: #eff6ff;
}

.stat-icon.orange {
  background: #fff7ed;
}

.stat-icon.red {
  background: #fef2f2;
}

.stat-icon.purple {
  background: #f5f3ff;
}

.stat-label {
  display: block;

  color: #64748b;

  font-size: 10px;
  font-weight: 800;

  letter-spacing: 0.7px;
}

.stat-value {
  display: block;

  margin-top: 5px;

  color: #111827;

  font-size: 27px;
  line-height: 1;
}

.stat-method {
  display: block;

  margin-top: 5px;

  color: #6d28d9;

  font-size: 25px;
}

.stat-card small {
  display: block;

  margin-top: 7px;

  color: #94a3b8;

  font-size: 11px;
}

/* =========================================
   COMMON PANEL
========================================= */

.panel {
  margin: 15px 42px;

  padding: 22px;

  background: white;

  border: 1px solid #e8ebf2;

  border-radius: 14px;

  box-shadow:
    0 5px 20px rgba(15,23,42,0.045);
}

.panel-title,
.section-heading {
  display: flex;
  align-items: center;
  justify-content: space-between;

  gap: 15px;

  margin-bottom: 18px;
}

.panel-title {
  justify-content: flex-start;
}

.title-icon {
  width: 39px;
  height: 39px;

  display: flex;
  align-items: center;
  justify-content: center;

  border-radius: 10px;

  background: #eef2ff;

  font-size: 18px;
}

.panel h2,
.section-heading h2 {
  margin: 0;

  color: #111827;

  font-size: 17px;
  font-weight: 750;
}

.panel-title p,
.section-heading p {
  margin: 4px 0 0;

  color: #64748b;

  font-size: 12px;
}

/* =========================================
   FILTERS
========================================= */

.filter-row {
  display: grid;

  grid-template-columns:
    minmax(250px, 2fr)
    repeat(3, minmax(130px, 1fr))
    auto;

  gap: 10px;
}

.search-box {
  height: 40px;

  display: flex;
  align-items: center;

  padding: 0 12px;

  border: 1px solid #dbe1ea;

  border-radius: 8px;

  background: #fff;
}

.search-box span {
  color: #94a3b8;
  margin-right: 8px;
}

.search-box input {
  width: 100%;

  border: 0;
  outline: 0;

  color: #172033;

  font-size: 12px;
}

.search-box input::placeholder {
  color: #94a3b8;
}

.filter-row select {
  height: 40px;

  padding: 0 10px;

  border: 1px solid #dbe1ea;

  border-radius: 8px;

  background: white;

  color: #475569;

  outline: none;

  font-size: 12px;
}

.filter-row select:focus {
  border-color: #818cf8;
}

.reset-button {
  height: 40px;

  padding: 0 15px;

  border: 1px solid #dbe1ea;

  border-radius: 8px;

  background: #f8fafc;

  color: #475569;

  font-size: 12px;
  font-weight: 700;
}

.reset-button:hover {
  background: #eef2f7;
}

.filter-footer {
  margin-top: 15px;

  display: flex;
  justify-content: space-between;

  color: #64748b;

  font-size: 11px;
}

.filter-footer strong {
  color: #334155;
}

/* =========================================
   MONITORING BAR
========================================= */

.monitoring-bar {
  margin: 15px 42px;

  display: flex;
  justify-content: space-between;
  align-items: center;

  padding: 12px 16px;

  border-radius: 9px;

  background: #ecfdf5;

  border: 1px solid #bbf7d0;

  color: #166534;

  font-size: 11px;
}

.monitoring-bar > div {
  display: flex;
  align-items: center;
  gap: 8px;
}

.status-dot {
  width: 7px;
  height: 7px;

  border-radius: 50%;

  background: #22c55e;
}

/* =========================================
   MAP
========================================= */

.map-panel {
  overflow: hidden;
}

.map-count {
  padding: 7px 11px;

  border-radius: 20px;

  background: #f1f5f9;

  color: #475569;

  font-size: 11px;
  font-weight: 700;
}

.map-wrapper {
  position: relative;

  height: 570px;

  overflow: hidden;

  border-radius: 11px;

  border: 1px solid #dce2eb;
}

.disaster-map {
  width: 100%;
  height: 100%;
}

.map-legend {
  position: absolute;

  z-index: 1000;

  right: 15px;
  bottom: 15px;

  min-width: 150px;

  padding: 12px;

  border-radius: 9px;

  background: rgba(255,255,255,0.95);

  box-shadow:
    0 4px 16px rgba(15,23,42,0.15);

  font-size: 10px;
}

.map-legend strong {
  display: block;

  margin-bottom: 7px;

  color: #334155;
}

.map-legend div {
  display: flex;

  align-items: center;

  gap: 7px;

  margin-top: 5px;

  color: #64748b;
}

.legend-dot {
  width: 8px;
  height: 8px;

  border-radius: 50%;
}

.legend-dot.critical {
  background: #ef4444;
}

.legend-dot.high {
  background: #f97316;
}

.legend-dot.medium {
  background: #eab308;
}

.legend-dot.low {
  background: #22c55e;
}

.legend-dot.quantum {
  background: #8b5cf6;
}

/* =========================================
   MAP POPUP
========================================= */

.map-popup {
  min-width: 265px;
}

.map-popup h3 {
  margin: 10px 0;

  color: #111827;

  font-size: 14px;

  line-height: 1.35;
}

.popup-header {
  display: flex;
  justify-content: space-between;
  align-items: center;

  color: #64748b;

  font-size: 10px;
}

.popup-priority {
  padding: 4px 7px;

  border-radius: 5px;

  color: white;

  font-weight: 800;
}

.popup-location {
  padding: 9px;

  border-radius: 7px;

  background: #f8fafc;

  color: #334155;

  font-size: 11px;
}

.popup-grid {
  display: grid;

  grid-template-columns: 1fr 1fr;

  gap: 7px;

  margin-top: 10px;
}

.popup-grid div {
  padding: 7px;

  border-radius: 6px;

  background: #f8fafc;
}

.popup-grid span {
  display: block;

  color: #94a3b8;

  font-size: 9px;
}

.popup-grid strong {
  display: block;

  margin-top: 3px;

  color: #334155;

  font-size: 11px;
}

.popup-details {
  margin-top: 8px;
}

.popup-details p {
  margin: 5px 0;

  color: #64748b;

  font-size: 10px;
}

.quantum-selected {
  margin-top: 9px;

  padding: 7px;

  border-radius: 6px;

  background: #f5f3ff;

  color: #6d28d9;

  text-align: center;

  font-size: 10px;
  font-weight: 800;
}

/* =========================================
   QUANTUM
========================================= */

.quantum-panel {
  background:
    linear-gradient(
      135deg,
      #ffffff,
      #faf8ff
    );
}

.success-badge {
  padding: 6px 10px;

  border-radius: 20px;

  background: #ecfdf5;

  color: #15803d;

  font-size: 10px;
  font-weight: 800;
}

.quantum-empty {
  padding: 45px 20px;

  text-align: center;
}

.large-quantum {
  width: 65px;
  height: 65px;

  margin: auto;

  display: flex;
  align-items: center;
  justify-content: center;

  border-radius: 50%;

  background: #f5f3ff;

  color: #7c3aed;

  font-size: 35px;
}

.quantum-empty h3 {
  margin: 15px 0 7px;

  font-size: 16px;
}

.quantum-empty p {
  margin: 0 auto 18px;

  max-width: 520px;

  color: #64748b;

  font-size: 12px;
}

.quantum-metrics {
  display: grid;

  grid-template-columns:
    repeat(6, minmax(0, 1fr));

  gap: 10px;
}

.quantum-metrics > div {
  padding: 15px;

  border-radius: 9px;

  background: white;

  border: 1px solid #e7e1f7;
}

.quantum-metrics span {
  display: block;

  color: #94a3b8;

  font-size: 9px;
  font-weight: 800;
}

.quantum-metrics strong {
  display: block;

  margin-top: 7px;

  color: #5b21b6;

  font-size: 18px;
}

.resource-section {
  margin-top: 20px;
}

.resource-section h3 {
  margin: 0 0 15px;

  font-size: 14px;
}

.resource-row {
  margin-bottom: 14px;
}

.resource-label {
  display: flex;
  justify-content: space-between;

  margin-bottom: 6px;

  color: #475569;

  font-size: 11px;
}

.resource-label strong {
  color: #172033;
}

.progress-track {
  width: 100%;
  height: 7px;

  overflow: hidden;

  border-radius: 20px;

  background: #e2e8f0;
}

.progress-fill {
  height: 100%;

  border-radius: 20px;

  background:
    linear-gradient(
      90deg,
      #8b5cf6,
      #4f46e5
    );
}

.resource-row small {
  display: block;

  margin-top: 4px;

  color: #94a3b8;

  font-size: 9px;
}

/* =========================================
   SCENARIO
========================================= */

.scenario-controls {
  display: grid;

  grid-template-columns:
    repeat(3, 1fr)
    1.5fr;

  gap: 12px;

  align-items: end;
}

.scenario-input label {
  display: block;

  margin-bottom: 7px;

  color: #475569;

  font-size: 11px;
  font-weight: 700;
}

.scenario-input input {
  width: 100%;
  height: 40px;

  padding: 0 10px;

  border: 1px solid #dbe1ea;

  border-radius: 8px;

  outline: none;
}

.scenario-input input:focus {
  border-color: #818cf8;
}

.scenario-button {
  height: 40px;

  border: 0;

  border-radius: 8px;

  color: white;

  background: #4f46e5;

  font-size: 11px;
  font-weight: 750;
}

.scenario-current {
  margin-top: 15px;

  padding: 11px 13px;

  border-radius: 8px;

  background: #f8fafc;

  color: #64748b;

  font-size: 11px;
}

.scenario-result {
  margin-top: 18px;

  padding: 18px;

  border-radius: 10px;

  background: #fafaff;

  border: 1px solid #e4e1f7;
}

.scenario-result h3 {
  margin: 0;

  font-size: 14px;
}

.scenario-result > p {
  color: #64748b;

  font-size: 11px;
}

.scenario-metrics {
  display: grid;

  grid-template-columns: repeat(3, 1fr);

  gap: 10px;
}

.scenario-metrics div {
  padding: 13px;

  border-radius: 8px;

  background: white;

  border: 1px solid #e5e7eb;
}

.scenario-metrics span {
  display: block;

  color: #94a3b8;

  font-size: 9px;
}

.scenario-metrics strong {
  display: block;

  margin-top: 5px;

  color: #4f46e5;

  font-size: 17px;
}

.scenario-status {
  margin-top: 13px;

  color: #64748b;

  font-size: 10px;
}

/* =========================================
   RECOMMENDATIONS
========================================= */

.recommendation-list {
  display: flex;

  flex-direction: column;

  gap: 10px;
}

.recommendation-card {
  display: flex;

  align-items: flex-start;

  gap: 13px;

  padding: 15px;

  border: 1px solid #e7e5f4;

  border-radius: 10px;

  background:
    linear-gradient(
      90deg,
      #faf9ff,
      #ffffff
    );
}

.recommendation-number {
  width: 31px;
  height: 31px;

  flex-shrink: 0;

  display: flex;
  align-items: center;
  justify-content: center;

  border-radius: 50%;

  background: #7c3aed;

  color: white;

  font-size: 12px;
  font-weight: 800;
}

.recommendation-main {
  flex: 1;
  min-width: 0;
}

.recommendation-title {
  color: #111827;

  font-size: 14px;
  font-weight: 800;
}

.recommendation-type {
  display: inline-block;

  margin-top: 4px;

  color: #64748b;

  font-size: 10px;
}

.recommendation-location {
  margin-top: 7px;

  color: #475569;

  font-size: 11px;
  font-weight: 650;
}

.recommendation-resources {
  margin-top: 7px;

  color: #64748b;

  font-size: 10px;
}

.recommendation-why {
  margin-top: 9px;

  color: #64748b;

  font-size: 10px;

  line-height: 1.5;
}

.recommendation-why strong {
  color: #475569;
}

.recommendation-score {
  min-width: 90px;

  padding-left: 15px;

  text-align: right;

  border-left: 1px solid #e5e7eb;
}

.recommendation-score span {
  display: block;

  color: #94a3b8;

  font-size: 8px;
  font-weight: 800;
}

.recommendation-score strong {
  display: block;

  margin-top: 5px;

  color: #6d28d9;

  font-size: 22px;
}

.no-recommendations {
  padding: 30px;

  text-align: center;

  color: #94a3b8;

  font-size: 12px;
}

/* =========================================
   DISTRIBUTION
========================================= */

.distribution-grid {
  display: grid;

  grid-template-columns:
    repeat(4, 1fr);

  gap: 12px;
}

.distribution-card {
  display: flex;

  align-items: center;

  gap: 12px;

  padding: 17px;

  border-radius: 10px;

  background: #f8fafc;

  border: 1px solid #e7ebf0;
}

.distribution-dot {
  width: 11px;
  height: 11px;

  border-radius: 50%;
}

.distribution-dot.critical {
  background: #ef4444;
}

.distribution-dot.high {
  background: #f97316;
}

.distribution-dot.medium {
  background: #eab308;
}

.distribution-dot.low {
  background: #22c55e;
}

.distribution-card span {
  display: block;

  color: #64748b;

  font-size: 10px;
}

.distribution-card strong {
  display: block;

  margin-top: 4px;

  font-size: 21px;
}

/* =========================================
   TABLE
========================================= */

.table-wrapper {
  width: 100%;

  overflow-x: auto;
}

table {
  width: 100%;

  border-collapse: collapse;

  font-size: 10px;
}

thead {
  background: #f8fafc;
}

th {
  padding: 11px 10px;

  text-align: left;

  color: #64748b;

  font-size: 9px;
  font-weight: 800;

  white-space: nowrap;
}

td {
  padding: 12px 10px;

  border-top: 1px solid #edf0f4;

  color: #64748b;

  vertical-align: middle;
}

.rank {
  width: 35px;

  color: #94a3b8;

  font-weight: 800;
}

.disaster-name {
  min-width: 190px;

  color: #172033;

  font-weight: 700;
}

.priority-value {
  color: #6d28d9;

  font-weight: 800;
}

.level-badge {
  display: inline-block;

  padding: 4px 7px;

  border-radius: 5px;

  font-size: 8px;
  font-weight: 800;
}

.level-badge.critical {
  background: #fef2f2;
  color: #dc2626;
}

.level-badge.high {
  background: #fff7ed;
  color: #ea580c;
}

.level-badge.medium {
  background: #fefce8;
  color: #ca8a04;
}

.level-badge.low {
  background: #f0fdf4;
  color: #16a34a;
}

.table-badge {
  padding: 6px 10px;

  border-radius: 6px;

  background: #f1f5f9;

  color: #64748b;

  font-size: 9px;
  font-weight: 800;
}

/* =========================================
   SOURCES
========================================= */

.sources-grid {
  margin: 15px 42px 25px;

  display: grid;

  grid-template-columns:
    repeat(3, 1fr);

  gap: 13px;
}

.source-card {
  display: flex;

  align-items: center;

  gap: 13px;

  padding: 17px;

  background: white;

  border: 1px solid #e8ebf2;

  border-radius: 11px;

  box-shadow:
    0 4px 15px rgba(15,23,42,0.035);
}

.source-icon {
  width: 39px;
  height: 39px;

  display: flex;
  align-items: center;
  justify-content: center;

  border-radius: 9px;

  background: #eef2ff;

  font-size: 18px;
}

.source-card h3 {
  margin: 0;

  color: #172033;

  font-size: 12px;
}

.source-card p {
  margin: 4px 0 0;

  color: #64748b;

  font-size: 10px;
}

/* =========================================
   FOOTER
========================================= */

footer {
  margin-top: 30px;

  padding: 30px 20px 35px;

  text-align: center;

  color: white;

  background:
    linear-gradient(
      135deg,
      #111827,
      #1e1b4b
    );
}

.footer-logo {
  width: 42px;
  height: 42px;

  margin: 0 auto 9px;

  display: flex;
  align-items: center;
  justify-content: center;

  border-radius: 12px;

  background: rgba(255,255,255,0.1);

  font-size: 23px;
}

footer h2 {
  margin: 0;

  font-size: 16px;
}

footer p {
  margin: 7px 0 0;

  color: #94a3b8;

  font-size: 10px;
}

/* =========================================
   LOADING
========================================= */

.loading-screen {
  min-height: 100vh;

  display: flex;
  justify-content: center;
  align-items: center;

  background:
    radial-gradient(
      circle at center,
      #eef2ff,
      #f8fafc 55%
    );
}

.loading-card {
  width: 360px;

  padding: 40px;

  text-align: center;

  background: white;

  border: 1px solid #e8ebf2;

  border-radius: 18px;

  box-shadow:
    0 15px 45px rgba(15,23,42,0.1);
}

.quantum-loader {
  width: 65px;
  height: 65px;

  margin: 0 auto 18px;

  display: flex;
  justify-content: center;
  align-items: center;

  border-radius: 50%;

  background: #f5f3ff;

  color: #7c3aed;

  font-size: 34px;

  animation: quantumPulse 1.5s infinite;
}

.loading-card h1 {
  margin: 0;

  font-size: 20px;
}

.loading-card p {
  color: #64748b;

  font-size: 12px;
}

@keyframes quantumPulse {
  0%,
  100% {
    transform: scale(1);
    box-shadow: 0 0 0 0 rgba(124,58,237,0.12);
  }

  50% {
    transform: scale(1.06);
    box-shadow: 0 0 0 15px rgba(124,58,237,0);
  }
}

/* =========================================
   RESPONSIVE
========================================= */

@media (max-width: 1100px) {

  .stats-section {
    grid-template-columns:
      repeat(2, 1fr);
  }

  .filter-row {
    grid-template-columns:
      1fr 1fr;
  }

  .quantum-metrics {
    grid-template-columns:
      repeat(3, 1fr);
  }

  .scenario-controls {
    grid-template-columns:
      repeat(2, 1fr);
  }

}

@media (max-width: 800px) {

  .top-header {
    padding: 18px 20px;

    flex-direction: column;

    align-items: flex-start;
  }

  .header-actions {
    width: 100%;
  }

  .quantum-main-button {
    flex: 1;
  }

  .stats-section {
    padding-left: 20px;
    padding-right: 20px;

    grid-template-columns: 1fr;
  }

  .panel {
    margin-left: 20px;
    margin-right: 20px;
  }

  .monitoring-bar {
    margin-left: 20px;
    margin-right: 20px;

    flex-direction: column;

    align-items: flex-start;

    gap: 7px;
  }

  .map-wrapper {
    height: 450px;
  }

  .quantum-metrics {
    grid-template-columns:
      repeat(2, 1fr);
  }

  .distribution-grid {
    grid-template-columns:
      repeat(2, 1fr);
  }

  .sources-grid {
    margin-left: 20px;
    margin-right: 20px;

    grid-template-columns: 1fr;
  }

}

@media (max-width: 550px) {

  .brand-area h1 {
    font-size: 20px;
  }

  .header-actions {
    flex-direction: column;

    align-items: stretch;
  }

  .live-badge {
    justify-content: center;
  }

  .filter-row {
    grid-template-columns: 1fr;
  }

  .filter-footer {
    flex-direction: column;

    gap: 5px;
  }

  .section-heading {
    flex-direction: column;

    align-items: flex-start;
  }

  .quantum-metrics {
    grid-template-columns: 1fr;
  }

  .scenario-controls {
    grid-template-columns: 1fr;
  }

  .scenario-metrics {
    grid-template-columns: 1fr;
  }

  .recommendation-card {
    flex-direction: column;
  }

  .recommendation-score {
    width: 100%;

    padding-left: 0;
    padding-top: 10px;

    text-align: left;

    border-left: 0;
    border-top: 1px solid #e5e7eb;
  }

  .distribution-grid {
    grid-template-columns: 1fr;
  }

  .map-legend {
    right: 8px;
    bottom: 8px;
  }

}


/* =========================================================
   DARK THEME OVERRIDES
   ========================================================= */

:root {
  color-scheme: dark;
}

html,
body {
  background: #07111f;
  color: #e5e7eb;
}

body {
  background:
    radial-gradient(circle at 15% 0%, rgba(79,70,229,0.12), transparent 28%),
    #07111f;
}

.app {
  background: transparent;
}

.top-header {
  background:
    radial-gradient(circle at 10% 0%, rgba(99,102,241,0.20), transparent 28%),
    linear-gradient(135deg, #0b1220, #111827 60%, #172554);
  border-bottom: 1px solid #1f2937;
}

.stats-section {
  background: transparent;
}

.stat-card,
.panel {
  background: #0d1726;
  border-color: #1f2b3d;
  box-shadow: 0 10px 28px rgba(0,0,0,0.18);
}

.stat-label,
.stat-card small,
.panel-title p,
.section-heading p,
.filter-footer,
.monitoring-bar,
.resource-label,
.resource-source {
  color: #94a3b8;
}

.stat-value,
.panel h2,
.section-heading h2,
.resource-section h3,
.scenario-result h3,
.disaster-name {
  color: #f8fafc;
}

.stat-method {
  color: #a78bfa;
}

.stat-icon.blue { background: rgba(59,130,246,0.14); }
.stat-icon.orange { background: rgba(249,115,22,0.14); }
.stat-icon.red { background: rgba(239,68,68,0.14); }
.stat-icon.purple { background: rgba(139,92,246,0.14); }

.search-box,
.filter-row select,
.scenario-input input {
  background: #0a1320;
  border-color: #26364b;
  color: #e5e7eb;
}

.search-box input {
  background: transparent;
  color: #e5e7eb;
}

.search-box input::placeholder {
  color: #64748b;
}

.filter-row select option {
  background: #0d1726;
  color: #e5e7eb;
}

.reset-button {
  background: #111c2c;
  border-color: #2a3a50;
  color: #cbd5e1;
}

.reset-button:hover {
  background: #172235;
}

.monitoring-bar {
  background: rgba(6,78,59,0.25);
  border-color: #14532d;
  color: #bbf7d0;
}

.map-wrapper {
  border-color: #26364b;
}

.map-count {
  background: #111c2c;
  color: #cbd5e1;
}

.map-legend {
  background: rgba(10,19,32,0.94);
  color: #cbd5e1;
  border: 1px solid #26364b;
}

.map-legend strong {
  color: #f8fafc;
}

.map-legend div {
  color: #94a3b8;
}

.map-popup {
  color: #e5e7eb;
}

.map-popup h3 {
  color: #111827;
}

.popup-location,
.popup-grid div {
  background: #f1f5f9;
}

.error-banner {
  background: rgba(127,29,29,0.25);
  border-color: #7f1d1d;
  color: #fecaca;
}

.quantum-panel {
  background:
    linear-gradient(135deg, #0d1726, #12102a);
  border-color: #312e81;
}

.quantum-metrics > div,
.scenario-metrics > div,
.distribution-card {
  background: #101b2b;
  border-color: #29384d;
}

.quantum-metrics span,
.scenario-metrics span,
.distribution-card span {
  color: #94a3b8;
}

.quantum-metrics strong,
.scenario-metrics strong {
  color: #c4b5fd;
}

.large-quantum {
  background: rgba(124,58,237,0.14);
}

.resource-status-panel {
  background:
    linear-gradient(135deg, #0d1726, #0b1b25);
  border-color: #164e63;
}

.resource-status-grid {
  display: grid;
  grid-template-columns: repeat(3, minmax(0, 1fr));
  gap: 12px;
}

.resource-status-grid > div {
  padding: 16px;
  border-radius: 10px;
  background: #101d2d;
  border: 1px solid #26364b;
}

.resource-status-grid span {
  display: block;
  color: #94a3b8;
  font-size: 10px;
  font-weight: 800;
  letter-spacing: .5px;
}

.resource-status-grid strong {
  display: block;
  margin-top: 7px;
  color: #f8fafc;
  font-size: 24px;
}

.inventory-badge {
  padding: 6px 10px;
  border-radius: 999px;
  font-size: 10px;
  font-weight: 800;
}

.inventory-badge.live {
  color: #86efac;
  background: rgba(34,197,94,0.12);
  border: 1px solid rgba(74,222,128,0.25);
}

.inventory-badge.configured {
  color: #fbbf24;
  background: rgba(245,158,11,0.10);
  border: 1px solid rgba(245,158,11,0.25);
}

.resource-source {
  margin-top: 12px;
  font-size: 10px;
}

.progress-track {
  background: #243246;
}

.recommendation-card,
.table-wrapper {
  background: #0f1a2a;
  border-color: #26364b;
}

.recommendation-title,
.recommendation-main strong,
.table-wrapper td {
  color: #e5e7eb;
}

.recommendation-location,
.recommendation-resources,
.recommendation-why,
.table-wrapper th {
  color: #94a3b8;
}

.table-wrapper tr {
  border-color: #26364b;
}

footer {
  background: #08111e !important;
  border-top-color: #1f2937 !important;
  color: #cbd5e1 !important;
}

footer h2 {
  color: #f8fafc !important;
}

footer p,
footer small {
  color: #64748b !important;
}

@media (max-width: 800px) {
  .resource-status-grid {
    grid-template-columns: 1fr;
  }
}
'''

Path("/content/frontend/src/App.css").write_text(
    app_css,
    encoding="utf-8"
)

print("✅ Complete App.css replaced")
print(
    "File size:",
    Path("/content/frontend/src/App.css").stat().st_size,
    "bytes"
)

In [ ]:
import subprocess

result = subprocess.run(
    ["npm", "run", "build"],
    cwd="/content/frontend",
    capture_output=True,
    text=True,
    timeout=180
)

print(result.stdout)
print(result.stderr)

if result.returncode != 0:
    raise RuntimeError("Frontend build failed.")

print("✅ FRONTEND BUILD SUCCESSFUL")


In [ ]:
import subprocess
import time
import requests
from pathlib import Path

FRONTEND_LOG = Path("/content/frontend.log")

frontend_running = False

try:
    response = requests.get(
        "http://127.0.0.1:5173",
        timeout=3
    )
    if response.status_code < 500:
        frontend_running = True
        print("✅ Frontend is already running on port 5173.")
except requests.exceptions.RequestException:
    print("Frontend is not running.")

if not frontend_running:
    print("Starting Vite frontend...")

    frontend_log = open(FRONTEND_LOG, "a")

    frontend_process = subprocess.Popen(
        [
            "npm",
            "run",
            "dev",
            "--",
            "--host",
            "0.0.0.0",
            "--port",
            "5173"
        ],
        cwd="/content/frontend",
        stdout=frontend_log,
        stderr=subprocess.STDOUT,
        start_new_session=True
    )

    print("Frontend process started.")
    print("PID:", frontend_process.pid)

    for attempt in range(20):
        try:
            response = requests.get(
                "http://127.0.0.1:5173",
                timeout=3
            )
            if response.status_code < 500:
                frontend_running = True
                print("✅ Vite frontend is ready.")
                break
        except requests.exceptions.RequestException:
            pass

        print(f"Waiting for Vite... {attempt + 1}/20")
        time.sleep(1)

if not frontend_running:
    print("\n❌ Frontend failed to start.")
    if FRONTEND_LOG.exists():
        print("".join(FRONTEND_LOG.read_text(errors="ignore").splitlines(True)[-40:]))
    raise RuntimeError("Vite frontend could not be started.")


In [ ]:
from google.colab import output

output.serve_kernel_port_as_iframe(
    5173,
    height=1100
)
print("✅ Dashboard opened on port 5173.")
